# 32 Transportabilitas Antarkursus dan Waktu


In [1]:
from pathlib import Path
import os, types
from IPython.display import display
PROJECT_ROOT='/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive',force_remount=False)
ROOT_32=Path(os.environ.get('TRANSPORT32_ROOT',PROJECT_ROOT)).expanduser().resolve()
if not (ROOT_32/'LIB').is_dir():raise FileNotFoundError('Folder LIB belum tersedia: '+str(ROOT_32))
print('Proyek:',ROOT_32)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Masukan dan batas evaluasi

- LMS: `OUTPUT/PRAPROSES/01_Praproses_main_LMS/02_dataset_behavioral_numeric.csv`
  untuk 6658 peserta dan `01_selected_userid_highest_activity.csv` untuk metadata
  kursus/waktu. Metadata ditautkan menurut user; seluruh 14546 baris selected
  tidak dijadikan kohort tambahan. Jumlah kursus final sumber:171.
- Pendikom: `DATA/compiled_pendikom_moodle_logs.csv`, kedua CSV praproses
  final di `OUTPUT/PRAPROSES/02_Praproses_Pendikom/`, dan
  `LIB/PROTOKOL/pendikom_ils_labels.csv`. Sumber 565 record,461 WITH_ILS,50 kursus.
- LIB dan sumber notebook 02,22,25,26,27 harus tersedia dengan fungsi ilmiah
  yang sama. Definisi fungsi/konstanta diperiksa; tidak menjalankan driver lama.
  Hasil fitting tahap 25–31 tidak menjadi dependensi untuk split baru ini.

Tidak mengunduh dataset pengganti. ZIP proyek terbaru tidak menyertakan DATA;
raw compiled perlu tetap ada di Drive. Sebelum eksperimen, rekonstruksi raw
divalidasi terhadap hasil praproses final pada split lama 322/139. Label ILS
raw wajib cocok dengan LIB. Verifikasi ini memastikan masukan yang konsisten
sebelum pembagian kursus dan waktu yang baru.

Pendikom mengukur kesesuaian terhadap ILS independen. LMS mengukur fidelity
terhadap teacher dari train dan diagnostik profil, bukan akurasi ILS.
Perbandingan initializer LMS terhadap CARE merupakan concordance diagnostik,
bukan pembuktian keunggulan CARE dengan menjadikannya ground truth.


## Protokol yang ditetapkan

| Pengujian | Split | Referensi |
|---|---|---|
| Pendikom antarkursus | GroupKFold 5, seluruh kursus dipisahkan | ILS held-out |
| LMS antarkursus | GroupKFold 5 pada 171 kursus | Frozen CARE train mapping |
| Pendikom waktu | Batas 1 Juli 2021 UTC; seluruh rentang observasi sebelum/sesudah | ILS kohort sesudah batas |
| LMS waktu | Audit metadata saja | Fitur lama memakai jendela P95 full-course; belum rekonstruksi forward-safe |

GroupKFold tidak memakai label ILS, dan setiap peserta mendapat satu prediksi
out-of-fold per metode/run. Lima fold sekitar 80/20 berbeda dari holdout 70/30
lama. Tidak melakukan uji berpasangan terhadap angka holdout lama.

Temporal Pendikom: train `last_activity < cutoff`, test `first_activity >= cutoff`.
Timestamp yang hilang/tidak sah dan record melintasi batas dikeluarkan dengan
alasan tercatat. Bila kursus train juga muncul pada test, seluruh peserta train
kursus tersebut dipurge. Minimal 100 train,30 test, dua kursus tiap sisi; jika tidak
layak, status temporal INELIGIBLE disimpan tanpa mencari tanggal pengganti.
Batas kalender ditetapkan sebelum melihat skor. Ini validasi retrospektif
seluruh record, bukan klaim prediksi dini atau hasil intervensi instruksional.

CARE-KM, Cao, Bai, BK-Modes masing-masing satu run; Random, Huang, K-Modes++, HR-2023
masing-masing 15 run. k=16, q=16, separation, min_gain 0, SMD, max_iter 100 tetap.
Default 11 konteks ×64 fit =704 fit attempts; jumlah mengikuti kelayakan temporal.
Setiap metode stokastik mendapat seed yang sama pada fold/run yang sama.
Seed, waktu, fold, dan identitas peserta dicatat; tidak memilih best run.


In [2]:
KONFIGURASI_32={
    'cohorts':['PENDIKOM','LMS'],
    'n_folds':5,
    'stochastic_runs':15,
    'bootstrap_repetitions':10000,
    'bootstrap_seed':32161600,
    'temporal_pendikom':True,
    'temporal_cutoff_utc':'2021-07-01T00:00:00Z',
    'classifiers':True,
    'force':False,
}


## Fitting hanya pada train

LMS mempelajari ulang positive quantiles, degeneracy dan sparse fallback dari
train tiap fold, dengan fungsi pipeline_data yang sama. Data kategorikal full
tidak langsung dipakai sebagai representasi holdout baru.

Pendikom mempelajari ulang acuan durasi Q75 per kursus dan global dari train.
Kursus test yang tidak dikenal memakai globalQ75 train, sesuai fallback asli.
Mask fitur konstan dipelajari dari train 19 kandidat; hasilnya bisa berbeda dari
mask 18 fitur pada split lama. Kamus evidence 19 fitur tetap bersumber notebook 22.
Seluruh threshold/mask/state disimpan. Metadata identitas/kursus/waktu dan ILS
tidak masuk predictors. Tidak melakukan tuning dari test.

Mapping utama mode-first, magnitudo evidence anggota dan pemecah seri tetap
memakai fungsi 26. Dua alternatif CARE member-only/member-first memakai partisi
yang sama; behavioral rules langsung dihitung sekali per konteks tanpa cluster.

Classifier CARE mode-first mengikuti notebook 22: LMS empat model binary per
keluarga; Pendikom satu model BaseLS per keluarga, kemudian diurai empat arah.
NB/DT/SVM mempertahankan konfigurasi dan representasi masing-masing kohort.
Tidak memakai target ILS untuk melatih classifier. Train unresolved dicatat
dan dikeluarkan hanya dari target training yang tidak tersedia. Satu kelas
train memakai constant predictor yang dinyatakan eksplisit; nol target berarti
abstention. Semua peserta test tetap disimpan, termasuk unresolved/failure.

Failure fit/metrik tidak diganti seed lain. Hasil failure disimpan pada cache
dan seluruh test untuk jalur itu dicatat sebagai abstention. Interupsi runtime
berbeda dari failure algoritma dan bisa dilanjutkan.


In [3]:
CAPSULES_32 = {'LMS': "import numpy as np\nimport pandas as pd\nMAIN_FEATURES=['content_visit','content_stay','selfass_visit_proxy','selfass_stay_proxy',\n 'exercise_visit_proxy','exercise_stay_proxy','forum_visit','forum_stay','forum_post',\n 'quiz_stay_results_proxy','navigation_overview_visit','navigation_overview_stay','return_to_overview_rate']\n\nDEGENERATE_MAX_SHARE=0.95\n\nFEATURE_PLAN = [\n    {'feature':'content_visit','mode':'GRAF_FIXED','low':75.0,'high':100.0},\n    {'feature':'content_stay','mode':'EMPIRICAL','low':50.0,'high':75.0},\n    {'feature':'selfass_visit_proxy','mode':'GRAF_FIXED','low':25.0,'high':75.0},\n    {'feature':'selfass_stay_proxy','mode':'EMPIRICAL','low':50.0,'high':75.0},\n    {'feature':'exercise_visit_proxy','mode':'GRAF_FIXED','low':25.0,'high':75.0},\n    {'feature':'exercise_stay_proxy','mode':'EMPIRICAL','low':50.0,'high':75.0},\n    {'feature':'forum_visit','mode':'GRAF_DEGENERACY','low':7.0,'high':50.0},\n    {'feature':'forum_stay','mode':'GRAF_DEGENERACY','low':5.0,'high':30.0},\n    {'feature':'forum_post','mode':'GRAF_SPARSE','low':1.0,'high':10.0},\n    {'feature':'quiz_stay_results_proxy','mode':'GRAF_FIXED','low':30.0,'high':60.0},\n    {'feature':'navigation_overview_visit','mode':'GRAF_DEGENERACY','low':10.0,'high':20.0},\n    {'feature':'navigation_overview_stay','mode':'EMPIRICAL','low':50.0,'high':75.0},\n    {'feature':'return_to_overview_rate','mode':'EMPIRICAL','low':np.nan,'high':np.nan},\n]\n\ndef classify_cut(x,lo,hi):\n    x=pd.to_numeric(x,errors='coerce').fillna(0).astype(float)\n    return pd.Series(np.where(x<lo,1,np.where(x<hi,2,3)),index=x.index,dtype='int8')\n\ndef positive_q(x):\n    x=pd.to_numeric(x,errors='coerce').fillna(0).astype(float); pos=x[x>0]\n    if pos.empty: raise RuntimeError('Positive quantile unavailable.')\n    lo=float(pos.quantile(1/3)); hi=float(pos.quantile(2/3))\n    if hi<=lo:\n        u=np.sort(pos.unique()); lo=float(np.quantile(u,1/3)); hi=float(np.quantile(u,2/3))\n    if hi<=lo: raise RuntimeError('Degenerate train threshold.')\n    return lo,hi\n\ndef share123(c): return c.value_counts(normalize=True).reindex([1,2,3],fill_value=0.0)\n\ndef fit_main_categorizer(train):\n    state={}; audit=[]\n    for sp in FEATURE_PLAN:\n        f=sp['feature']; mode=sp['mode']; x=train[f]; lo=sp['low']; hi=sp['high']; method=mode\n        if mode=='EMPIRICAL': lo,hi=positive_q(x)\n        elif mode=='GRAF_DEGENERACY':\n            c0=classify_cut(x,lo,hi); sh=share123(c0)\n            if sh.max()>DEGENERATE_MAX_SHARE or (sh==0).any(): lo,hi=positive_q(x); method='GRAF_FALLBACK_POSITIVE_Q33_Q67'\n        elif mode=='GRAF_SPARSE':\n            c0=classify_cut(x,lo,hi); sh=share123(c0)\n            if sh.max()>DEGENERATE_MAX_SHARE or (sh==0).any():\n                pos=pd.to_numeric(x,errors='coerce').fillna(0); pos=pos[pos>0]\n                lo=0.0; hi=float(pos.median()) if len(pos) else 1.0; method='ZERO_AWARE_POSITIVE_MEDIAN'\n        state[f]={'low':float(lo),'high':float(hi),'method':method}\n        audit.append({'feature':f,**state[f]})\n    return state,pd.DataFrame(audit)\n\ndef apply_main_categorizer(df,state):\n    out=pd.DataFrame({'user':df['user'].astype(str)})\n    for f in MAIN_FEATURES:\n        st=state[f]; x=pd.to_numeric(df[f],errors='coerce').fillna(0).astype(float)\n        if st['method']=='ZERO_AWARE_POSITIVE_MEDIAN':\n            out[f]=np.where(x<=0,1,np.where(x<st['high'],2,3)).astype(int)\n        else:\n            out[f]=classify_cut(x,st['low'],st['high']).astype(int)\n    return out\n", 'PEND': 'from pathlib import Path\nimport csv\nimport numpy as np\nimport pandas as pd\nPENDIKOM_FEATURES = [\n    \'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\n    \'exercise_visit\',\'exercise_stay\',\'ques_detail\',\'ques_overview\',\n    \'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\n    \'ques_interpret\',\'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\n    \'navigation_skip\',\'navigation_overview_visit\',\'navigation_overview_stay\',\n]\n\nPENDIKOM_THRESHOLDS = {\n    \'content_visit\': (75,100),\n    \'content_stay\': (50,75),\n    \'selfass_visit\': (25,75),\n    \'selfass_twice_wrong\': (25,50),\n    \'exercise_visit\': (25,75),\n    \'exercise_stay\': (50,75),\n    \'ques_detail\': (50,75),\n    \'ques_overview\': (50,75),\n    \'ques_facts\': (50,75),\n    \'ques_concepts\': (50,75),\n    \'ques_graphics\': (50,75),\n    \'ques_text\': (50,75),\n    \'ques_interpret\': (50,75),\n    \'ques_develop\': (50,75),\n    \'quiz_revisions\': (20,50),\n    \'quiz_stay_results\': (30,60),\n    \'navigation_skip\': (1,2),\n    \'navigation_overview_visit\': (10,20),\n    \'navigation_overview_stay\': (50,75),\n}\n\nILS_CODE_COLS = [\n    \'ils_active_reflective_code\',\n    \'ils_sensing_intuitive_code\',\n    \'ils_visual_verbal_code\',\n    \'ils_sequential_global_code\',\n]\n\ndef read_compiled_pendikom(path):\n    """Read normal multi-column CSV or the repository one-column wrapped CSV."""\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(path)\n\n    with path.open(\'r\', encoding=\'utf-8-sig\', newline=\'\') as fh:\n        outer = list(csv.reader(fh))\n\n    if outer and all(len(r) == 1 for r in outer):\n        inner = [next(csv.reader([r[0]])) for r in outer]\n        width = len(inner[0])\n        bad = [i for i, r in enumerate(inner) if len(r) != width]\n        if bad:\n            raise RuntimeError(f\'Wrapped CSV has inconsistent row widths: {bad[:10]}\')\n        frame = pd.DataFrame(inner[1:], columns=inner[0])\n    else:\n        frame = pd.read_csv(path, low_memory=False)\n\n    frame.columns = [str(c).strip() for c in frame.columns]\n\n    text_cols = {\n        \'record_group\',\'username_anon\',\'idnumber_anon\',\'course_fullname\',\n        \'ils_course_fullname\',\'ils_active_reflective_code\',\n        \'ils_sensing_intuitive_code\',\'ils_visual_verbal_code\',\n        \'ils_sequential_global_code\',\'ils_ls_class\',\'ils_semantic_label\',\n    } | {f\'Q{i:02d}\' for i in range(1,45)}\n\n    for c in frame.columns:\n        if c not in text_cols:\n            frame[c] = pd.to_numeric(frame[c], errors=\'coerce\')\n\n    return frame\n\ndef _fit_q75_reference(train_frame, seconds_col, opportunity_col=None):\n    cols = [\'courseid\', seconds_col] + ([opportunity_col] if opportunity_col else [])\n    t = train_frame[cols].copy()\n    if opportunity_col is not None:\n        t = t[t[opportunity_col].gt(0)]\n    positive = t[t[seconds_col].gt(0)]\n\n    by_course = positive.groupby(\'courseid\')[seconds_col].quantile(0.75)\n    global_q75 = float(positive[seconds_col].quantile(0.75)) if len(positive) else 1.0\n    if not np.isfinite(global_q75) or global_q75 <= 0:\n        global_q75 = 1.0\n    return by_course, global_q75\n\ndef _ratio_pct(num, den):\n    n = pd.to_numeric(num, errors=\'coerce\').fillna(0).to_numpy(float)\n    d = pd.to_numeric(den, errors=\'coerce\').fillna(0).to_numpy(float)\n    out = np.zeros(len(n), dtype=float)\n    mask = d > 0\n    out[mask] = 100.0 * n[mask] / d[mask]\n    return out\n\ndef build_pendikom_numeric(frame):\n    b = frame\n    meta = {\n        \'userid\': b[\'userid\'].astype(int),\n        \'courseid\': b[\'courseid\'].astype(int),\n        \'total_activity\': pd.to_numeric(b[\'total_activity\'], errors=\'raise\').astype(int),\n        \'active_days\': pd.to_numeric(b[\'active_days\'], errors=\'raise\').astype(int),\n    }\n    if \'course_fullname\' in b.columns:\n        meta[\'course_fullname\'] = b[\'course_fullname\'].astype(str)\n\n    out = pd.DataFrame(meta)\n    out[\'content_visit\'] = _ratio_pct(\n        b[\'content_visit_evidence_count\'], b[\'content_available\']\n    )\n\n    ref_by_course, ref_global = stay_refs[\'content\']\n    ref = b[\'courseid\'].map(ref_by_course).fillna(ref_global).to_numpy(float)\n    sec = pd.to_numeric(b[\'content_stay_seconds\'], errors=\'coerce\').fillna(0).to_numpy(float)\n    out[\'content_stay\'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)\n\n    out[\'selfass_visit\'] = _ratio_pct(\n        b[\'question_attempt_count\'], b[\'question_available\']\n    )\n    out[\'selfass_twice_wrong\'] = _ratio_pct(\n        b[\'twice_wrong_question_count\'], b[\'repeated_question_count\']\n    )\n    out[\'exercise_visit\'] = _ratio_pct(\n        b[\'submitted_assignment_count\'], b[\'assign_available\']\n    )\n\n    ref_by_course, ref_global = stay_refs[\'exercise\']\n    ref = b[\'courseid\'].map(ref_by_course).fillna(ref_global).to_numpy(float)\n    sec = pd.to_numeric(b[\'exercise_stay_seconds\'], errors=\'coerce\').fillna(0).to_numpy(float)\n    out[\'exercise_stay\'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)\n\n    for tag in [\'detail\',\'overview\',\'facts\',\'concepts\',\'graphics\',\'text\',\'interpret\',\'develop\']:\n        numerator = (\n            \'interpret_question_score_sum\'\n            if tag == \'interpret\'\n            else f\'ques_{tag}_fraction_sum\'\n        )\n        out[f\'ques_{tag}\'] = _ratio_pct(\n            b[numerator], b[f\'ques_{tag}_attempted_count\']\n        )\n\n    out[\'quiz_revisions\'] = _ratio_pct(\n        b[\'revised_question_attempt_count\'], b[\'question_attempt_count\']\n    )\n\n    events = pd.to_numeric(\n        b[\'quiz_results_event_count\'], errors=\'coerce\'\n    ).fillna(0).to_numpy(float)\n    seconds = pd.to_numeric(\n        b[\'quiz_results_stay_seconds_sum\'], errors=\'coerce\'\n    ).fillna(0).to_numpy(float)\n    q = np.zeros(len(b), dtype=float)\n    mask = events > 0\n    q[mask] = seconds[mask] / events[mask]\n    out[\'quiz_stay_results\'] = q\n\n    module_views = np.maximum(\n        pd.to_numeric(b[\'module_view_count\'], errors=\'coerce\').fillna(0).to_numpy(float),\n        1.0,\n    )\n    out[\'navigation_skip\'] = (\n        100.0 *\n        pd.to_numeric(b[\'navigation_skip_count\'], errors=\'coerce\').fillna(0).to_numpy(float)\n        / module_views\n    )\n    out[\'navigation_overview_visit\'] = (\n        100.0 *\n        pd.to_numeric(b[\'overview_visit_count\'], errors=\'coerce\').fillna(0).to_numpy(float)\n        / module_views\n    )\n\n    ref_by_course, ref_global = stay_refs[\'overview\']\n    ref = b[\'courseid\'].map(ref_by_course).fillna(ref_global).to_numpy(float)\n    sec = pd.to_numeric(b[\'overview_stay_seconds\'], errors=\'coerce\').fillna(0).to_numpy(float)\n    out[\'navigation_overview_stay\'] = np.where(sec > 0, 100.0 * sec / ref, 0.0)\n\n    return out\n\ndef categorize_pendikom(numeric_frame):\n    meta_cols = [\n        c for c in [\'userid\',\'courseid\',\'course_fullname\',\'total_activity\',\'active_days\']\n        if c in numeric_frame.columns\n    ]\n    out = numeric_frame[meta_cols].copy()\n    for feature in PENDIKOM_FEATURES:\n        low, high = PENDIKOM_THRESHOLDS[feature]\n        v = pd.to_numeric(numeric_frame[feature], errors=\'coerce\').fillna(0).to_numpy(float)\n        out[feature] = np.where(\n            v < low, 1, np.where(v < high, 2, 3)\n        ).astype(np.int8)\n    return out\n', 'FACTORY': "import pandas as pd\nfrom sklearn.preprocessing import OneHotEncoder\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.naive_bayes import CategoricalNB\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.svm import SVC\nSPLIT_SEED=161600\ndef one_hot_encoder():\n    try:\n        return OneHotEncoder(handle_unknown='ignore', sparse_output=True)\n    except TypeError:\n        return OneHotEncoder(handle_unknown='ignore', sparse=True)\n\ndef categorical_nb():\n    try:\n        return CategoricalNB(alpha=1.0, min_categories=4)\n    except TypeError:\n        return CategoricalNB(alpha=1.0)\n\ndef build_main_classifiers():\n    # Prospective main-LMS configuration.\n    return {\n        'NB': categorical_nb(),\n        'DT': Pipeline([('onehot', one_hot_encoder()), ('model', DecisionTreeClassifier(random_state=SPLIT_SEED))]),\n        'SVM': Pipeline([('onehot', one_hot_encoder()), ('model', SVC(kernel='rbf', C=1.0, gamma='scale'))]),\n    }\n\ndef build_external_classifiers():\n    # Frozen configuration used by the locked ground-truth Table 2.\n    return {\n        'NB': categorical_nb(),\n        'DT': DecisionTreeClassifier(random_state=SPLIT_SEED, class_weight='balanced'),\n        'SVM': Pipeline([('onehot', one_hot_encoder()), ('svc', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced'))]),\n    }\n\ndef pendikom_semantic_rules():\n    rows = []\n    def add(feature, dimension, low, high):\n        rows.append({\n            'feature': feature,\n            'dimension': dimension,\n            'low_supports': low,\n            'high_supports': high,\n        })\n\n    for f in ['selfass_visit','selfass_twice_wrong','exercise_visit','exercise_stay']:\n        add(f, 'Active/Reflective', 'Reflective', 'Active')\n    for f in ['content_visit','content_stay','quiz_stay_results']:\n        add(f, 'Active/Reflective', 'Active', 'Reflective')\n\n    for f in ['selfass_visit','exercise_visit','ques_detail','ques_facts','quiz_revisions','quiz_stay_results']:\n        add(f, 'Sensing/Intuitive', 'Intuitive', 'Sensing')\n    for f in ['content_visit','content_stay','ques_concepts','ques_develop']:\n        add(f, 'Sensing/Intuitive', 'Sensing', 'Intuitive')\n\n    add('ques_graphics', 'Visual/Verbal', 'Verbal', 'Visual')\n    for f in ['content_visit','ques_text']:\n        add(f, 'Visual/Verbal', 'Visual', 'Verbal')\n\n    add('ques_detail', 'Sequential/Global', 'Global', 'Sequential')\n    for f in ['ques_overview','ques_interpret','ques_develop',\n              'navigation_skip','navigation_overview_visit','navigation_overview_stay']:\n        add(f, 'Sequential/Global', 'Sequential', 'Global')\n\n    return pd.DataFrame(rows)\n", 'SEM': 'import itertools\nimport numpy as np\nimport pandas as pd\nMETHODS = [\'Random\',\'Huang\',\'Cao\',\'Bai\',\'K-Modes++\',\'HR-2023\',\'CARE-KM\',\'BK-Modes\']\n\nSTOCHASTIC = {\'Random\',\'Huang\',\'K-Modes++\',\'HR-2023\'}\n\nORDER = [\'CARE-KM\',\'Cao\',\'Bai\',\'BK-Modes\',\'K-Modes++\',\'Random\',\'Huang\',\'HR-2023\']\n\nKEYS = [\'active_reflective\',\'sensing_intuitive\',\'visual_verbal\',\'sequential_global\']\n\nDIMS = [\'Active/Reflective\',\'Sensing/Intuitive\',\'Visual/Verbal\',\'Sequential/Global\']\n\nPOLES = [(\'Active\',\'Reflective\'),(\'Sensing\',\'Intuitive\'),(\'Visual\',\'Verbal\'),(\'Sequential\',\'Global\')]\n\nABBR = {\'Active\':\'Act.\',\'Reflective\':\'Ref.\',\'Sensing\':\'Sens.\',\'Intuitive\':\'Int.\',\n        \'Visual\':\'Vis.\',\'Verbal\':\'Ver.\',\'Sequential\':\'Seq.\',\'Global\':\'Gl.\'}\n\nLS_LOOKUP = {combo:\'LS%02d\' % (i+1) for i,combo in enumerate(itertools.product(*POLES))}\n\nMAIN = [\'content_visit\',\'content_stay\',\'selfass_visit_proxy\',\'selfass_stay_proxy\',\'exercise_visit_proxy\',\n        \'exercise_stay_proxy\',\'forum_visit\',\'forum_stay\',\'forum_post\',\'quiz_stay_results_proxy\',\n        \'navigation_overview_visit\',\'navigation_overview_stay\',\'return_to_overview_rate\']\n\nPEND = [\'content_visit\',\'content_stay\',\'selfass_visit\',\'selfass_twice_wrong\',\'exercise_visit\',\'exercise_stay\',\n        \'ques_detail\',\'ques_overview\',\'ques_facts\',\'ques_concepts\',\'ques_graphics\',\'ques_text\',\'ques_interpret\',\n        \'ques_develop\',\'quiz_revisions\',\'quiz_stay_results\',\'navigation_overview_visit\',\'navigation_overview_stay\']\n\ndef expected_rules(dataset):\n    rows=[]\n    def add(features,dim,low,high):\n        for f in features.split(): rows.append((f,dim,low,high))\n    if dataset==\'LMS\':\n        add(\'content_visit content_stay selfass_stay_proxy forum_visit quiz_stay_results_proxy\',DIMS[0],*POLES[0])\n        add(\'selfass_visit_proxy exercise_visit_proxy exercise_stay_proxy forum_post\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay\',DIMS[1],*POLES[1])\n        add(\'selfass_visit_proxy selfass_stay_proxy exercise_visit_proxy quiz_stay_results_proxy\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit forum_visit forum_stay forum_post\',DIMS[2],*POLES[2])\n        add(\'navigation_overview_visit navigation_overview_stay return_to_overview_rate\',DIMS[3],*POLES[3])\n    else:\n        add(\'selfass_visit selfass_twice_wrong exercise_visit exercise_stay\',DIMS[0],*POLES[0][::-1])\n        add(\'content_visit content_stay quiz_stay_results\',DIMS[0],*POLES[0])\n        add(\'selfass_visit exercise_visit ques_detail ques_facts quiz_revisions quiz_stay_results\',DIMS[1],*POLES[1][::-1])\n        add(\'content_visit content_stay ques_concepts ques_develop\',DIMS[1],*POLES[1])\n        add(\'ques_graphics\',DIMS[2],*POLES[2][::-1]); add(\'content_visit ques_text\',DIMS[2],*POLES[2])\n        add(\'ques_detail\',DIMS[3],*POLES[3][::-1])\n        add(\'ques_overview ques_interpret ques_develop navigation_overview_visit navigation_overview_stay\',DIMS[3],*POLES[3])\n    return sorted(rows)\n\ndef interpret(X,modes,labels,features,rules):\n    """Same mode-first rule and member-derived 11-unit magnitude as stages 11/13/22."""\n    sizes=np.bincount(labels,minlength=16);mapping=[];evidence=[];fpos={f:i for i,f in enumerate(features)}\n    prepared=[]\n    for dim,poles in zip(DIMS,POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);low=np.array([1 if v==poles[0] else -1 for v in rr.low_supports])\n        high=np.array([1 if v==poles[0] else -1 for v in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',low,np.where(data==\'3\',high,0))\n        mv=modes[:,cols];modevote=np.where(mv==\'1\',low,np.where(mv==\'3\',high,0))\n        prepared.append((rr,vote,modevote))\n    for cid in range(16):\n        item={\'cluster_id\':cid,\'cluster_name\':f\'C{cid+1:02d}\',\'cluster_size\':int(sizes[cid])};poles_selected=[];sources=[];conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(DIMS,POLES,KEYS)):\n            rr,vote,modevote=prepared[di];mv=modevote[cid];nv=vote[labels==cid]\n            mp,mn,mz=[int(np.sum(mv==v)) for v in [1,-1,0]]\n            pos,neg,neu=[int(np.sum(nv==v)) for v in [1,-1,0]]\n            ms,ns=mp-mn,pos-neg;selected=1 if ms>0 else -1 if ms<0 else 1 if ns>0 else -1 if ns<0 else 0\n            source=\'MODE\' if ms else \'MEMBER\' if ns else \'BALANCED\';sources.append(source)\n            total=pos+neg+neu\n            units=min(11,max(0,int(np.floor(11*(pos+.5*neu)/total+.5)))) if total else None\n            raw=2*units-11 if units is not None else None;mag=abs(raw) if raw is not None else None\n            category=\'Balanced\' if mag in (1,3) else \'Moderate preference\' if mag in (5,7) else \'Strong preference\' if mag in (9,11) else \'Unresolved\'\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));conflicts+=conflict\n            item[key]=short;item[key+\'_score\']=selected*mag if selected and mag is not None else np.nan\n            item[key+\'_category\']=category;item[key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            item[key+\'_resolved\']=bool(selected)\n            evidence.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(sizes[cid]),\'mapped_features\':len(rr),\n                \'mode_positive\':mp,\'mode_negative\':mn,\'mode_neutral\':mz,\'mode_score\':ms,\'member_positive\':pos,\'member_negative\':neg,\n                \'member_neutral\':neu,\'member_score\':ns,\'selected_pole\':short,\'decision_source\':source,\n                \'positive_adjusted_evidence\':pos+.5*neu,\'negative_adjusted_evidence\':neg+.5*neu,\n                \'positive_11eq_units\':units,\'negative_11eq_units\':11-units if units is not None else None,\n                \'member_11eq_raw_score\':raw,\'preference_11eq_magnitude\':mag,\'preference_category\':category,\n                \'mode_member_consistency\':not conflict,\'strong_mode_member_conflict\':conflict and mag in (9,11)})\n        resolved=all(p is not None for p in poles_selected)\n        item.update(ls_class=LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\',\n                    semantic_label=\'-\'.join(item[k] for k in KEYS),preference_signature=\'|\'.join(item[k+\'_code\'] for k in KEYS),\n                    mapping_status=\'RESOLVED\' if resolved else \'UNRESOLVED\',mode_dimensions=sources.count(\'MODE\'),\n                    member_dimensions=sources.count(\'MEMBER\'),balanced_dimensions=sources.count(\'BALANCED\'),mode_member_disagreements=conflicts)\n        mapping.append(item)\n    return pd.DataFrame(mapping),pd.DataFrame(evidence)\n\ndef profiles(mapping,labels,ids,indices=None):\n    cols=KEYS+[k+\'_score\' for k in KEYS]+[\'ls_class\',\'mapping_status\']\n    frame=mapping.set_index(\'cluster_id\').loc[labels,cols].reset_index()\n    frame.insert(0,\'identity\',ids)\n    if indices is not None: frame.insert(1,\'source_row_index\',indices)\n    return frame\n\ndef numeric_profiles(frame):\n    return np.column_stack([np.where(frame[k].eq(ABBR[p[0]]),1,np.where(frame[k].eq(ABBR[p[1]]),-1,0)) for k,p in zip(KEYS,POLES)])\n\ndef count_metrics(counts):\n    """2 x 3 confusion counts; column 3 is abstention, never dropped."""\n    a,b,c,d,e,f=np.moveaxis(np.asarray(counts,float),-1,0)\n    pos=a+b+c;neg=d+e+f;total=pos+neg\n    with np.errstate(divide=\'ignore\',invalid=\'ignore\'):\n        accuracy=(a+e)/total;ba=.5*(a/pos+e/neg)\n        denp=2*a+d+b+c;denn=2*e+b+d+f\n        fp=np.divide(2*a,denp,out=np.zeros_like(a),where=denp>0)\n        fn=np.divide(2*e,denn,out=np.zeros_like(e),where=denn>0)\n    return np.stack([accuracy,ba,.5*(fp+fn)],axis=-1)\n\ndef confusion_masks(truth,prediction):\n    # prediction: run x learner x dimension; output learner x run x dimension x six cells.\n    t=truth[None,:,:];p=prediction\n    conditions=[(t==1)&(p==1),(t==1)&(p==-1),(t==1)&(p==0),\n                (t==-1)&(p==1),(t==-1)&(p==-1),(t==-1)&(p==0)]\n    return np.stack(conditions,axis=-1).transpose(1,0,2,3).astype(float)\n', 'PROF': 'import numpy as np\nimport pandas as pd\nPRIMARY=\'PRIMARY_MODE_FIRST\'\n\nDIRECT=\'DIRECT_BEHAVIORAL_RULES\'\n\nMEMBER=\'MEMBER_ONLY\'\n\nCONSISTENT=\'DIRECTION_CONSISTENT_MEMBER_FIRST\'\n\ndef make_mapping(s,primary,evidence,rule):\n    """Change only the direction operator; reuse frozen P/N/Z and magnitude."""\n    if rule not in [MEMBER,CONSISTENT]:raise ValueError(\'Operator tidak didukung.\')\n    mapping=primary.copy();mapping[\'reported_member_disagreements\']=0;decisions=[]\n    for cid in range(16):\n        poles_selected=[];sources=[];reported=0;mode_conflicts=0\n        for di,(dim,poles,key) in enumerate(zip(s.DIMS,s.POLES,s.KEYS)):\n            e=evidence.iloc[cid*4+di];ms=int(e.mode_score);ns=int(e.member_score)\n            selected=int(np.sign(ns)) if ns else (int(np.sign(ms)) if rule==CONSISTENT else 0)\n            source=\'MEMBER\' if ns else \'MODE\' if selected else \'BALANCED\';sources.append(source)\n            mag=int(e.preference_11eq_magnitude)\n            if mag not in {1,3,5,7,9,11}:raise RuntimeError(\'Magnitudo sumber tidak valid.\')\n            pole=poles[0] if selected==1 else poles[1] if selected==-1 else None\n            short=s.ABBR[pole] if pole else \'Balanced\';poles_selected.append(pole)\n            conflict=bool(selected and ns and selected!=np.sign(ns));underlying=bool(ms*ns<0)\n            reported+=conflict;mode_conflicts+=underlying\n            mapping.loc[cid,key]=short;mapping.loc[cid,key+\'_score\']=selected*mag if selected else np.nan\n            mapping.loc[cid,key+\'_resolved\']=bool(selected);mapping.loc[cid,key+\'_code\']=short+str(mag) if selected else \'Balanced\'\n            decisions.append({\'cluster_id\':cid,\'dimension\':dim,\'cluster_size\':int(e.cluster_size),\'mode_score\':ms,\'member_score\':ns,\n                \'selected_pole\':short,\'decision_source\':source,\'magnitude\':mag,\'preference_category\':e.preference_category,\n                \'mode_member_conflict\':underlying,\'reported_member_conflict\':conflict,\n                \'strong_reported_member_conflict\':conflict and mag in (9,11),\'rule\':rule})\n        resolved=all(p is not None for p in poles_selected)\n        mapping.loc[cid,\'ls_class\']=s.LS_LOOKUP[tuple(poles_selected)] if resolved else \'AMBIGUOUS\'\n        mapping.loc[cid,\'semantic_label\']=\'-\'.join(mapping.loc[cid,k] for k in s.KEYS)\n        mapping.loc[cid,\'preference_signature\']=\'|\'.join(mapping.loc[cid,k+\'_code\'] for k in s.KEYS)\n        mapping.loc[cid,\'mapping_status\']=\'RESOLVED\' if resolved else \'UNRESOLVED\'\n        mapping.loc[cid,\'mode_dimensions\']=sources.count(\'MODE\');mapping.loc[cid,\'member_dimensions\']=sources.count(\'MEMBER\')\n        mapping.loc[cid,\'balanced_dimensions\']=sources.count(\'BALANCED\');mapping.loc[cid,\'mode_member_disagreements\']=mode_conflicts\n        mapping.loc[cid,\'reported_member_disagreements\']=reported\n    return mapping,pd.DataFrame(decisions)\n\ndef direct_profiles(s,X,ids,features,rules,indices=None):\n    n=len(X);frame=pd.DataFrame({\'identity\':ids,\'cluster_id\':[None]*n});evidence=[];codes=[]\n    if indices is not None:frame.insert(1,\'source_row_index\',indices)\n    fpos={f:i for i,f in enumerate(features)}\n    for dim,key,poles in zip(s.DIMS,s.KEYS,s.POLES):\n        rr=rules.loc[rules.dimension.eq(dim)];cols=np.array([fpos[f] for f in rr.feature]);lo=np.array([1 if p==poles[0] else -1 for p in rr.low_supports]);hi=np.array([1 if p==poles[0] else -1 for p in rr.high_supports])\n        data=X[:,cols];vote=np.where(data==\'1\',lo,np.where(data==\'3\',hi,0))\n        pos=(vote==1).sum(axis=1);neg=(vote==-1).sum(axis=1);neu=(vote==0).sum(axis=1);balance=pos-neg;direction=np.sign(balance).astype(int);codes.append(direction)\n        units=np.floor(11*(pos+.5*neu)/(pos+neg+neu)+.5).astype(int);mag=np.abs(2*units-11)\n        short=np.where(direction==1,s.ABBR[poles[0]],np.where(direction==-1,s.ABBR[poles[1]],\'Balanced\'))\n        frame[key]=short;frame[key+\'_score\']=np.where(direction!=0,direction*mag,np.nan)\n        frame[key+\'_category\']=np.where(np.isin(mag,[1,3]),\'Balanced\',np.where(np.isin(mag,[5,7]),\'Moderate preference\',\'Strong preference\'))\n        frame[key+\'_code\']=np.where(direction!=0,np.char.add(short.astype(str),mag.astype(str)),\'Balanced\')\n        evidence.append(pd.DataFrame({\'identity\':ids,\'dimension\':dim,\'positive\':pos,\'negative\':neg,\'neutral\':neu,\'balance\':balance,\n            \'selected_pole\':short,\'magnitude\':mag,\'resolved\':direction!=0,\'rule\':DIRECT}))\n    full=np.column_stack(codes);resolved=(full!=0).all(axis=1)\n    frame[\'mapping_status\']=np.where(resolved,\'RESOLVED\',\'UNRESOLVED\')\n    frame[\'ls_class\']=[s.LS_LOOKUP[tuple(poles[0] if value==1 else poles[1] for value,poles in zip(row,s.POLES))] if ok else \'AMBIGUOUS\' for row,ok in zip(full,resolved)]\n    return frame,pd.concat(evidence,ignore_index=True)\n', 'BK': '"""Python reimplementation of Algorithms 8–9, arXiv:2502.07081v2.\n\nThis module returns initial modes only. The final K-Modes optimization is\ndelegated to the unchanged optimizer of the host experiment pipeline.\nPaper-unspecified tie/empty/cap policies are explicit in IMPLEMENTATION_POLICY.\n"""\nimport numpy as np\n\nBK_IMPLEMENTATION_VERSION = \'bk-modes-2025-python-1.0.0\'\nPAPER_URL = \'https://arxiv.org/abs/2502.07081v2\'\nIMPLEMENTATION_POLICY = {\n    \'source\': PAPER_URL,\n    \'publication_status\': \'arXiv preprint, v2, 15 February 2025\',\n    \'implementation\': \'Python reimplementation; not author-supplied C code\',\n    \'cluster_selection\': \'largest within-leaf sum of SMD; first-created leaf on a tie\',\n    \'two_modes_initial_centers\': \'mode of selected leaf and first farthest source row\',\n    \'two_modes_assignment_tie\': \'second child, as in Algorithm 9\',\n    \'initial_categorical_mode_tie\': \'lexicographically smallest string\',\n    \'updated_categorical_mode_tie\': \'retain previous category if modal; otherwise lexicographic\',\n    \'farthest_row_tie\': \'first in the preserved source-row order\',\n    \'duplicate_rows\': \'retained with their original frequencies\',\n    \'identical_leaf\': \'not bisected; select another splittable leaf\',\n    \'empty_child\': \'move first farthest row from a non-singleton donor, then update modes; logged\',\n    \'inner_stopping\': \'child memberships unchanged; explicit iteration cap raises an error\',\n    \'final_kmodes\': \'host care_km.kmodes_fit, unchanged; host tie and empty-cluster rules\',\n    \'timing_scope\': \'all leaf selection, bisection, and mode construction belong to initialization\',\n    \'number_of_centers\': \'k leaf modes; coincident leaf modes are reported, not silently deduplicated\',\n    \'determinism\': \'fixed implementation, string categories, input order, and parameters\',\n}\n\n\ndef _validate_X(X):\n    X = np.asarray(X, dtype=object)\n    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:\n        raise ValueError(\'X harus berupa matriks kategorikal dengan baris dan fitur yang tidak kosong.\')\n    if any(not isinstance(value, (str, np.str_)) for value in X.flat):\n        raise TypeError(\'Kategori X harus berupa string yang sudah disiapkan secara konsisten.\')\n    if any(value == \'\' for value in X.flat):\n        raise ValueError(\'Kategori kosong harus ditangani sebelum inisialisasi.\')\n    return X\n\n\ndef _smd(X, centers):\n    """An n-by-k SMD matrix; avoid an n-by-k-by-d temporary."""\n    result = np.zeros((len(X), len(centers)), dtype=np.int64)\n    for feature in range(X.shape[1]):\n        result += X[:, feature, None] != centers[None, :, feature]\n    return result\n\n\ndef _mode(X, previous=None):\n    center = []\n    for feature in range(X.shape[1]):\n        values, counts = np.unique(X[:, feature], return_counts=True)\n        winners = values[counts == counts.max()]\n        old = previous[feature] if previous is not None else None\n        center.append(old if old in winners else winners[0])\n    return np.asarray(center, dtype=object)\n\n\ndef _assign_two(X, centers):\n    distances = _smd(X, centers)\n    # The strict comparison and second-child tie implement Algorithm 9.\n    labels = np.where(distances[:, 0] < distances[:, 1], 0, 1)\n    return labels.astype(np.int64), distances\n\n\ndef _two_modes(X, max_iter):\n    first = _mode(X)\n    distances = _smd(X, first[None, :])[:, 0]\n    farthest = int(np.argmax(distances))\n    if int(distances[farthest]) == 0:\n        raise ValueError(\'Leaf dengan satu vektor unik tidak dapat dibelah.\')\n    centers = np.asarray([first, X[farthest].copy()], dtype=object)\n    previous_labels = None\n    repairs = 0\n    history = []\n    for iteration in range(1, max_iter + 1):\n        labels, matrix = _assign_two(X, centers)\n        counts = np.bincount(labels, minlength=2)\n        for empty in np.flatnonzero(counts == 0):\n            assigned_distance = matrix[np.arange(len(X)), labels]\n            candidates = [int(i) for i in np.argsort(-assigned_distance, kind=\'stable\')\n                          if counts[labels[i]] > 1]\n            if not candidates:\n                raise RuntimeError(\'Tidak ada donor untuk memperbaiki child kosong.\')\n            selected = candidates[0]\n            counts[labels[selected]] -= 1\n            labels[selected] = int(empty)\n            counts[empty] += 1\n            repairs += 1\n        centers = np.asarray([_mode(X[labels == child], centers[child]) for child in range(2)], dtype=object)\n        objective = int(_smd(X, centers)[np.arange(len(X)), labels].sum())\n        history.append(objective)\n        if len(history) > 1 and history[-1] > history[-2]:\n            raise RuntimeError(\'Objective Two-Modes meningkat pada pembaruan.\')\n        if previous_labels is not None and np.array_equal(labels, previous_labels):\n            return {\'labels\': labels, \'centers\': centers, \'iterations\': iteration,\n                    \'empty_repairs\': repairs, \'objective_history\': history,\n                    \'initial_farthest_local_row\': farthest}\n        previous_labels = labels.copy()\n    raise RuntimeError(f\'Two-Modes belum konvergen setelah {max_iter} iterasi; hasil parsial tidak dipakai.\')\n\n\ndef bk_modes_init(X, k, max_iter=100, return_details=False):\n    """Return k initial leaf modes without running final global K-Modes.\n\n    ``return_details=True`` also returns the leaf memberships and split trace.\n    Categories must be explicit strings, as in the host pipeline datasets.\n    The source paper leaves several implementation choices unspecified;\n    those choices are recorded in IMPLEMENTATION_POLICY rather than hidden.\n    """\n    X = _validate_X(X)\n    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1:\n        raise ValueError(\'k harus berupa bilangan bulat positif.\')\n    if isinstance(max_iter, bool) or not isinstance(max_iter, (int, np.integer)) or max_iter < 1:\n        raise ValueError(\'max_iter harus berupa bilangan bulat positif.\')\n    k, max_iter = int(k), int(max_iter)\n    unique_rows = len({tuple(row) for row in X})\n    if k > unique_rows:\n        raise ValueError(f\'k={k} melebihi jumlah vektor kategorikal unik ({unique_rows}).\')\n\n    def node(identifier, indices, center=None):\n        center = _mode(X[indices]) if center is None else np.asarray(center, dtype=object).copy()\n        cost = int(_smd(X[indices], center[None, :]).sum())\n        return {\'node_id\': int(identifier), \'indices\': indices, \'center\': center, \'cost\': cost}\n\n    root = node(0, np.arange(len(X), dtype=np.int64))\n    leaves = {0: root}\n    next_id = 1\n    trace = []\n    for step in range(1, k):\n        splittable = [leaf for leaf in leaves.values() if leaf[\'cost\'] > 0]\n        if not splittable:\n            raise RuntimeError(\'Jumlah leaf yang diminta tidak tercapai.\')\n        parent = min(splittable, key=lambda leaf: (-leaf[\'cost\'], leaf[\'node_id\']))\n        local = _two_modes(X[parent[\'indices\']], max_iter)\n        children = []\n        for child in range(2):\n            indices = parent[\'indices\'][local[\'labels\'] == child]\n            if not len(indices):\n                raise RuntimeError(\'Two-Modes menghasilkan child kosong.\')\n            children.append(node(next_id + child, indices, local[\'centers\'][child]))\n        after_cost = sum(child[\'cost\'] for child in children)\n        if after_cost > parent[\'cost\']:\n            raise RuntimeError(\'Biseksi meningkatkan jumlah SMD dalam leaf.\')\n        trace.append({\'step\': step, \'parent_node\': parent[\'node_id\'],\n                      \'parent_rows\': int(len(parent[\'indices\'])), \'parent_cost\': parent[\'cost\'],\n                      \'child_nodes\': [child[\'node_id\'] for child in children],\n                      \'child_rows\': [int(len(child[\'indices\'])) for child in children],\n                      \'child_costs\': [child[\'cost\'] for child in children],\n                      \'parent_mode\': parent[\'center\'].tolist(),\n                      \'farthest_source_row\': int(parent[\'indices\'][local[\'initial_farthest_local_row\']]),\n                      \'two_modes_iterations\': local[\'iterations\'],\n                      \'empty_repairs\': local[\'empty_repairs\'],\n                      \'objective_history\': local[\'objective_history\']})\n        del leaves[parent[\'node_id\']]\n        for child in children:\n            leaves[child[\'node_id\']] = child\n        next_id += 2\n    ordered = [leaves[key] for key in sorted(leaves)]\n    modes = np.asarray([leaf[\'center\'] for leaf in ordered], dtype=object)\n    labels = np.full(len(X), -1, dtype=np.int64)\n    for label, leaf in enumerate(ordered):\n        labels[leaf[\'indices\']] = label\n    if np.any(labels < 0) or len(ordered) != k:\n        raise RuntimeError(\'Keanggotaan leaf belum lengkap.\')\n    details = {\'modes\': modes, \'leaf_labels\': labels, \'split_trace\': trace,\n               \'initial_global_mode_cost\': root[\'cost\'],\n               \'leaf_partition_cost\': sum(leaf[\'cost\'] for leaf in ordered),\n               \'initial_assignment_cost\': int(_smd(X, modes).min(axis=1).sum()),\n               \'unique_initial_modes\': len({tuple(row) for row in modes}),\n               \'total_two_modes_iterations\': sum(row[\'two_modes_iterations\'] for row in trace),\n               \'empty_repairs\': sum(row[\'empty_repairs\'] for row in trace),\n               \'version\': BK_IMPLEMENTATION_VERSION}\n    return details if return_details else modes\n', 'UTILS': "from pathlib import Path\nimport ast,gzip,hashlib,importlib.metadata,io,json,platform,sys,uuid\nimport numpy as np\nimport pandas as pd\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef digest(value):\n    return sha(json.dumps(clean(value), sort_keys=True, ensure_ascii=False, separators=(',', ':'), allow_nan=False).encode('utf-8'))\n\ndef clean(value):\n    if isinstance(value, dict):\n        return {str(k): clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [clean(v) for v in value]\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, np.bool_):\n        return bool(value)\n    return value\n\ndef atomic(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + '.' + uuid.uuid4().hex[:8] + '.tmp')\n    temp.write_bytes(data)\n    temp.replace(path)\n\ndef save_json(path, value):\n    atomic(path, json.dumps(clean(value), ensure_ascii=False, indent=2, allow_nan=False).encode('utf-8'))\n\ndef save_csv(path, frame):\n    data = frame.to_csv(index=False, lineterminator='\\n', float_format='%.17g').encode('utf-8')\n    atomic(path, gzip.compress(data, mtime=0) if str(path).endswith('.gz') else data)\n\ndef read_checkpoint(path):\n    try:\n        obj = json.loads(Path(path).read_text(encoding='utf-8'))\n        return obj if isinstance(obj, dict) else {}\n    except (OSError, ValueError, UnicodeError):\n        return {}\n\nclass Files:\n\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        self.cache = {}\n\n    def path(self, relative):\n        path = (self.root / relative).resolve()\n        if not path.is_relative_to(self.root):\n            raise ValueError('Lokasi di luar proyek: ' + relative)\n        return path\n\n    def read(self, relative):\n        path = self.path(relative)\n        notebook = path.suffix.lower() == '.ipynb'\n        # Colab may save cell outputs while the same notebook is being read.\n        # Wait for a complete, stable snapshot; never ignore code differences.\n        for attempt in range(10 if notebook else 1):\n            try:\n                before = path.stat()\n                stamp = (before.st_size, before.st_mtime_ns, before.st_ctime_ns)\n                if relative in self.cache and self.cache[relative][0] == stamp:\n                    return self.cache[relative][1]\n                data = path.read_bytes()\n                after = path.stat()\n                if stamp != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):\n                    raise RuntimeError('Berkas berubah ketika dibaca: ' + relative)\n                if notebook:\n                    nb = json.loads(data.decode('utf-8-sig'))\n                    if not isinstance(nb, dict) or not isinstance(nb.get('cells'), list):\n                        raise ValueError('Struktur notebook belum lengkap: ' + relative)\n                self.cache[relative] = (stamp, data)\n                return data\n            except (OSError, RuntimeError, UnicodeError, ValueError):\n                if not notebook or attempt == 9:\n                    raise\n                import time\n                time.sleep(min(.1 * 2 ** attempt, .8))\n\n    def hash(self, relative):\n        return sha(self.read(relative))\n\n    def frame(self, relative, strings=False):\n        data = self.read(relative)\n        if relative.endswith('.gz'):\n            data = gzip.decompress(data)\n        return pd.read_csv(io.BytesIO(data), dtype=str if strings else None, keep_default_na=not strings, float_precision='round_trip')\n\ndef runtime():\n    packages = {}\n    for name in ['numpy', 'pandas', 'scipy', 'sklearn', 'matplotlib']:\n        mod = sys.modules.get(name)\n        packages[name] = getattr(mod, '__version__', None) or importlib.metadata.version('scikit-learn' if name == 'sklearn' else name)\n    return {'python': platform.python_version(), 'packages': packages}\n\ndef notebook_hash(files, stem, tag, variable):\n    nb = json.loads(files.read('PYTHON/' + stem + '.ipynb').decode('utf-8-sig'))\n    code = []\n    for cell in nb['cells']:\n        if cell['cell_type'] != 'code':\n            continue\n        source = ''.join(cell['source'])\n        if tag in cell.get('metadata', {}).get('tags', []):\n            tree = ast.parse(source)\n            if len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assign) or len(tree.body[0].targets) != 1 or (not isinstance(tree.body[0].targets[0], ast.Name)) or (tree.body[0].targets[0].id != variable) or (not isinstance(ast.literal_eval(tree.body[0].value), dict)):\n                raise RuntimeError('Sel konfigurasi hanya boleh berisi literal ' + variable)\n        else:\n            code.append(source)\n    return digest(code)\n\ndef cache_read(files, path, signature, force=False):\n    record = read_checkpoint(path)\n    checksum = record.pop('record_sha256', '')\n    try:\n        valid = digest(record) == checksum\n    except (TypeError, ValueError):\n        valid = False\n    if force or not valid or record.get('signature') != signature or (record.get('status') != 'selesai'):\n        return None\n    if not all((files.path(p).is_file() and files.hash(p) == h for p, h in record.get('outputs', {}).items())):\n        return None\n    return record\n\ndef cache_write(files, path, signature, outputs, extra):\n    record = {'status': 'selesai', 'signature': signature, 'outputs': {str(p.relative_to(files.root)).replace('\\\\', '/'): sha(p.read_bytes()) for p in outputs}, **extra}\n    record['record_sha256'] = digest(record)\n    save_json(path, record)\n    return record\n\ndef unchanged(files, watched, code_checks):\n    for path, checksum in watched.items():\n        if files.hash(path) != checksum:\n            raise RuntimeError('Masukan berubah selama eksekusi: ' + path)\n    for stem, tag, var, expected in code_checks:\n        if notebook_hash(files, stem, tag, var) != expected:\n            raise RuntimeError('Kode notebook berubah selama eksekusi: ' + stem)\n\nFiles.sha = Files.hash\n", 'PINS': [{'path': 'LIB/pipeline_data.py', 'literal': None, 'definitions': {'MAIN_FEATURES': 'f6b25924a205cbe259f391ea5b26a5e5047b46c045992d66614e7c1566af5895', 'DEGENERATE_MAX_SHARE': '5a99370a88a54e0cd67667e385f09b6d1775e52a01f17a95da42feef212bdefb', 'FEATURE_PLAN': '02cf0c9dc6c47dbdc5fedafffea07d95213e34381d74baaf647b9445c5eb792f', 'classify_cut': '9d91e78dcbc9185c836bd8cd603555cab63cb001197db040f9e9a9c548fa4ec1', 'positive_q': '64f33aee4a56093ee7abb58863b0ebc91213bb231b10b19560f0991dbebfdc5f', 'share123': '674295773d7895485a174f7f86ba16981a50ea469bceb7bb75fee703a00e186e', 'fit_main_categorizer': '02a822ec517a4a83e0eb5fa375c90d266559476952d7141112403e51fa507aef', 'apply_main_categorizer': 'f180f867c494960a0e8977b513d10b69c79bdcaa5d725db822cc25b76746c869'}}, {'path': 'PYTHON/02_Praproses_Pendikom.ipynb', 'literal': None, 'definitions': {'PENDIKOM_FEATURES': '7daa4cca58b52f2b472cc477bc0ef148fdddab443e236dc844d52b76ff049c6e', 'PENDIKOM_THRESHOLDS': '80ee2420784ca8a948d1388db6e9059ed5b15c30be8ac551896df94c19488a70', 'ILS_CODE_COLS': '623851673db6562e30a765b7585e1f5cd2b0a5f83d6b54b8648dfa935a719a8a', 'read_compiled_pendikom': '42904c4348f860afb88a9f864ab8ad7ebd13c42ae8bc08429a602fc30e746e57', '_fit_q75_reference': '0328f9c0fff3c67f92a8805888cb84f627c7f2d6ce095d2c93b10406ea1d6b39', '_ratio_pct': '8c474022bc9890a656bb770cd00edf909bbb89da3c05a8086b945bf42490adac', 'build_pendikom_numeric': 'c2efb55fee5d53c5c142c69f52943787974d78781ad982c65719b4852a59b597', 'categorize_pendikom': 'd7a6faf81a815c01bf1ac7e8fb3dd0ebf3744e63a756a211c0d3610becec8151'}}, {'path': 'PYTHON/22_Holdout_LMS_Pendikom.ipynb', 'literal': None, 'definitions': {'one_hot_encoder': '37ecc01094e335834ee40f42de8813bbde41e048a85f8a20f1c2af44e11c0246', 'categorical_nb': '7ff452e172275c135f632b420bd5ce7e4d1c561c01cd256284f7632f6dcafb9c', 'build_main_classifiers': '7665145d4c831379bc8f5ec478df9d7910ce44ad19cb9210b341f5afd24f5ecd', 'build_external_classifiers': '6e5b1315c8cbe0b061d02512799ee553dd667a434f64916fa3204b7743074d81', 'pendikom_semantic_rules': '362a00341870c2c993c790e764692d9dafd093d62316d4d957b9cc49e3c382ed'}}, {'path': 'PYTHON/26_Pengaruh_Initializer_Profil_Semantik.ipynb', 'literal': 'SEMANTIC26_SOURCE', 'definitions': {'METHODS': '22fad5c61bc74dcef82aa15cde71aa05bfaa8f7c4dc573196b160ab965d08713', 'STOCHASTIC': 'aa00579f35bd52d94f49c72f88f1fa9f0d015e7a1896dc3d1834adaf82de4937', 'ORDER': '1a14a2d71c25bdc24575d8f55e93d357438242746db659cfc20c71b940f016f8', 'KEYS': '647ff39d3d927008e11e1b147189336c22319778d308ac1e2a31e4acaa28890e', 'DIMS': 'd7a214570318ce7986d9d408e917034ca3febdc1abb6597f78b3b021080f29b5', 'POLES': '7a4d8a58734f53b2772a1de9cf76e37078f6eac44d648c0d55e75984d8229ca4', 'ABBR': '345490c3ad55b85222accf2c5b5bdfe6f61503340f80e4d3672add073ba34dc5', 'LS_LOOKUP': 'e898460a3af9da3c8e7b0c86c2ddb885b569efb9d8c12deb857f1beaafd6a756', 'MAIN': '7ac5525b6e37a62b4bcbfe4e3fb45dd709a1694ee7417710086239a31d9d5e04', 'PEND': '47a41ce686f56660134d2f6712d7766d703c4e1b7a3c88a0274116329637a73d', 'expected_rules': 'e363ea9472639393fdf17a3f3c90490266be15e688a588490a418ba6a29eb537', 'interpret': '86a02a1dde500c95edf0efeaf03f9049ecb40eab5a6c4693c1af2561100d4b9e', 'profiles': 'affa620d5880b06a6792b678aa1150321b1b32447bbccbb07ed047433c4fc8f7', 'numeric_profiles': '30eceb47def2a7621a92522c6c39ac9ae18dd934bfe326f728cf24c34302b46d', 'count_metrics': 'a6efb28f4f009e8f3cfc8a4b37695a72b977ad656be3690483b09eb85d5367d0', 'confusion_masks': '6e5926c54dbc310e81fdf7b84547c1f09bb27d93e4a25e689a6064973bcaf373'}}, {'path': 'PYTHON/27_Perbandingan_Pembentukan_Profil.ipynb', 'literal': 'PROFILE27_SOURCE', 'definitions': {'PRIMARY': '61e5cc6ef08b070f2edabfa71c3db4c88b02088aca02ecbd79bcf9c9617f9186', 'DIRECT': '23d6f0473be593325b937dc1a564947810b123e7bf339e3484b023df1a9c826f', 'MEMBER': 'b4f46bf743bec9ed1970c50a932fad2f6082f3de48b62adb502c0b42e0e98e5b', 'CONSISTENT': '527f714b503cf0a4b3244bdfb4d384f7a0254114b010ea3d67161c5e2eb6ad16', 'make_mapping': 'c4d7942eb7c4560b495c2b7a434e98db69058d1e734f8db76a580c206a7f2466', 'direct_profiles': '0b17951410781968ef3d9ec63ab3e4bbf98872cbef6317487c75b842d3fbe9aa'}}, {'path': 'PYTHON/25_Perbandingan_Initializer_Terintegrasi.ipynb', 'literal': 'BK_MODES_SOURCE', 'sha256': '4340c434adc901ad866c5c14a074804f17e97739cd13761953aa707cbaee0c56'}, {'path': 'LIB/care_km.py', 'sha256': '80046c658fd7f9462e987fc5ece73241c6bc0fa094305895549a7c4b455c8628'}]}


In [4]:
RUNTIME32_SOURCE = '"""Course-disjoint and whole-window temporal validation; frozen scientific kernels."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport ast,csv,gzip,io,json,time,types,uuid\nimport numpy as np\nimport pandas as pd\n\nSTAGE=\'32_Transportabilitas_Antarkursus_dan_Waktu\'\nLMS=\'OUTPUT/PRAPROSES/01_Praproses_main_LMS/\'\nPEND=\'OUTPUT/PRAPROSES/02_Praproses_Pendikom/\'\nRAW=\'DATA/compiled_pendikom_moodle_logs.csv\'\nREF=\'LIB/PROTOKOL/pendikom_ils_labels.csv\'\nDEFAULT={\'cohorts\':[\'PENDIKOM\',\'LMS\'],\'n_folds\':5,\'stochastic_runs\':15,\'bootstrap_repetitions\':10000,\n    \'bootstrap_seed\':32161600,\'temporal_pendikom\':True,\'temporal_cutoff_utc\':\'2021-07-01T00:00:00Z\',\n    \'classifiers\':True,\'force\':False}\nCARE={\'candidate_rule\':\'separation\',\'candidate_budget\':16,\'min_gain\':0.0}\nMETRICS=[\'accuracy\',\'balanced_accuracy\',\'macro_f1\']\n\ndef utc():return datetime.now(timezone.utc).isoformat()\n\ndef config(configuration,s):\n    if set(configuration)-set(DEFAULT):raise ValueError(\'Konfigurasi32 tidak dikenal.\')\n    c={**DEFAULT,**configuration};c[\'cohorts\']=list(c[\'cohorts\'])\n    if not c[\'cohorts\'] or len(set(c[\'cohorts\']))!=len(c[\'cohorts\']) or set(c[\'cohorts\'])-{\'LMS\',\'PENDIKOM\'}:raise ValueError(\'cohorts harus LMS/PENDIKOM.\')\n    for key,low,high in [(\'n_folds\',2,5),(\'stochastic_runs\',1,15),(\'bootstrap_repetitions\',100,100000),(\'bootstrap_seed\',0,2**32-1)]:\n        if isinstance(c[key],bool) or not isinstance(c[key],int) or not low<=c[key]<=high:raise ValueError(key+\' di luar rentang.\')\n    for key in [\'temporal_pendikom\',\'classifiers\',\'force\']:\n        if not isinstance(c[key],bool):raise ValueError(key+\' harus True/False.\')\n    t=pd.Timestamp(c[\'temporal_cutoff_utc\'])\n    if t.tzinfo is None:raise ValueError(\'Batas waktu wajib memiliki zona waktu; gunakan UTC Z.\')\n    c[\'temporal_cutoff_utc\']=t.tz_convert(\'UTC\').isoformat();return c\n\ndef load_module(text,name,filename=None):\n    m=types.ModuleType(name);m.__file__=str(filename or name);exec(compile(text,str(m.__file__),\'exec\'),m.__dict__);return m\n\ndef definitions(text):\n    result={}\n    for n in ast.parse(text).body:\n        key=n.name if isinstance(n,ast.FunctionDef) else n.targets[0].id if isinstance(n,ast.Assign) and isinstance(n.targets[0],ast.Name) else None\n        if key:result[key]=ast.dump(n)\n    return result\n\ndef source_code(files,path):\n    value=files.read(path).decode(\'utf-8-sig\')\n    if path.endswith(\'.ipynb\'):\n        nb=json.loads(value);return \'\\n\\n\'.join(\'\'.join(c[\'source\']) for c in nb[\'cells\'] if c[\'cell_type\']==\'code\')\n    return value\n\ndef verify_sources(u,files,capsules):\n    watched={};evidence=[]\n    for pin in capsules[\'PINS\']:\n        text=source_code(files,pin[\'path\'])\n        if pin.get(\'literal\'):\n            found=[ast.literal_eval(n.value) for n in ast.parse(text).body if isinstance(n,ast.Assign) and len(n.targets)==1 and isinstance(n.targets[0],ast.Name) and n.targets[0].id==pin[\'literal\']]\n            if len(found)!=1:raise RuntimeError(\'Sumber tertanam tidak ditemukan: \'+pin[\'path\'])\n            text=found[0]\n        if \'definitions\' in pin:\n            actual=definitions(text)\n            for name,want in pin[\'definitions\'].items():\n                if u.sha(actual.get(name,\'\').encode())!=want:raise RuntimeError(\'Fungsi/konstanta sumber berubah: \'+pin[\'path\']+\'/\'+name)\n        elif u.sha(text.encode())!=pin[\'sha256\']:raise RuntimeError(\'Implementasi sumber berbeda: \'+pin[\'path\'])\n        if pin[\'path\'].endswith(\'.py\'):watched[pin[\'path\']]=files.hash(pin[\'path\'])\n        evidence.append({**pin,\'verified\':True})\n    return watched,evidence\n\ndef timestamps(series):\n    x=pd.to_numeric(series,errors=\'coerce\');x=x.where(x.gt(100000000))\n    x=x.where(x.lt(1e12),x/1000)\n    missing=x.isna()\n    if missing.any():\n        text=pd.to_datetime(series[missing],utc=True,errors=\'coerce\',format=\'mixed\')\n        vals=text.map(lambda v:v.timestamp() if pd.notna(v) else np.nan);x.loc[missing]=vals\n    return x.astype(float)\n\ndef raw_snapshot(files):\n    rows=list(csv.reader(io.StringIO(files.read(RAW).decode(\'utf-8-sig\'))))\n    if rows and all(len(r)==1 for r in rows):rows=[next(csv.reader([r[0]])) for r in rows]\n    if not rows or len(set(rows[0]))!=len(rows[0]) or any(len(r)!=len(rows[0]) for r in rows):raise RuntimeError(\'CSV compiled tidak konsisten.\')\n    return pd.DataFrame(rows[1:],columns=rows[0])\n\ndef ils_codes(s,reference):\n    values=[]\n    for key,poles in zip(s.KEYS,s.POLES):\n        c=reference[\'ils_\'+key+\'_code\'].astype(str);a=c.str.startswith(s.ABBR[poles[0]]);b=c.str.startswith(s.ABBR[poles[1]])\n        if not (a^b).all():raise RuntimeError(\'Arah ILS tidak dikenal: \'+key)\n        values.append(np.where(a,1,-1))\n    arr=np.column_stack(values)\n    expected=[s.LS_LOOKUP[tuple(p[0] if v==1 else p[1] for v,p in zip(row,s.POLES))] for row in arr]\n    if reference.ils_ls_class.astype(str).tolist()!=expected:raise RuntimeError(\'Base LS ILS tidak cocok dengan empat arah.\')\n    return arr\n\ndef fit_refs(p,train):\n    return {name:p._fit_q75_reference(train,seconds,opp) for name,seconds,opp in [\n        (\'content\',\'content_stay_seconds\',\'content_available\'),(\'exercise\',\'exercise_stay_seconds\',\'assign_available\'),(\'overview\',\'overview_stay_seconds\',None)]}\n\ndef selection_audit(u,out,raw):\n    rows=[]\n    for flag,g in raw.groupby(\'has_ils\'):\n        item={\'has_ils\':str(flag),\'n_source_records\':len(g),\'n_unique_users\':g.userid.nunique(),\'n_courses\':g.courseid.nunique()}\n        for col in [\'total_activity\',\'active_days\']:\n            x=pd.to_numeric(g[col],errors=\'coerce\');item.update({col+\'_median\':x.median(),col+\'_p25\':x.quantile(.25),col+\'_p75\':x.quantile(.75),col+\'_missing\':int(x.isna().sum())})\n        rows.append(item)\n    u.save_csv(out/\'AUDIT/01_seleksi_ILS_565_461.csv\',pd.DataFrame(rows))\n    counts=raw.groupby([\'courseid\',\'has_ils\']).size().unstack(fill_value=0).reset_index();u.save_csv(out/\'AUDIT/02_ketersediaan_ILS_per_kursus.csv\',counts)\n\ndef prepare(u,files,out,capsules,modules,cfg):\n    s,p,l,f=modules[\'SEM\'],modules[\'PEND\'],modules[\'LMS\'],modules[\'FACTORY\']\n    watched,pins=verify_sources(u,files,capsules)\n    paths=[LMS+\'02_dataset_behavioral_numeric.csv\',LMS+\'01_selected_userid_highest_activity.csv\',\n        PEND+\'01_numeric_behavioral_pendikom.csv\',PEND+\'02_categorical_behavioral_pendikom.csv\',RAW,REF]\n    missing=[v for v in paths if not files.path(v).is_file()]\n    if missing:raise FileNotFoundError(\'Masukan tahap32 belum tersedia; gunakan sumber asli yang sama:\\n\'+\'\\n\'.join(missing))\n    watched.update({v:files.hash(v) for v in paths})\n    main=files.frame(paths[0],True)\n    if list(main)!=[\'user\']+s.MAIN or len(main)!=6658 or main.user.duplicated().any():raise RuntimeError(\'Kohort/fitur LMS final berubah.\')\n    for c in s.MAIN:main[c]=pd.to_numeric(main[c],errors=\'raise\').astype(float)\n    if not np.isfinite(main[s.MAIN].to_numpy()).all():raise RuntimeError(\'Fitur LMS tidak finite.\')\n    metadata=[\'user\',\'course\',\'first_t\',\'last_t\',\'course_start\',\'course_end\',\'window_source\',\'n_courses_user\']\n    selected=pd.read_csv(io.BytesIO(files.read(paths[1])),dtype=str,usecols=metadata,keep_default_na=False)\n    if selected.user.duplicated().any():raise RuntimeError(\'Metadata selected memiliki user berulang.\')\n    main=main.merge(selected,on=\'user\',how=\'left\',validate=\'one_to_one\',indicator=True)\n    if not main._merge.eq(\'both\').all():raise RuntimeError(\'Metadata kursus LMS tidak lengkap untuk 6658 peserta.\')\n    main=main.drop(columns=\'_merge\').rename(columns={\'user\':\'identity\'});main[\'course\']=main.course.astype(str)\n    for c in [\'first_t\',\'last_t\',\'course_start\',\'course_end\']:main[c]=timestamps(main[c])\n    raw=raw_snapshot(files);selection_audit(u,out,raw)\n    source=p.read_compiled_pendikom(files.path(RAW));source=source.loc[source.has_ils.eq(1)].copy()\n    if len(source)!=461 or source.userid.duplicated().any():raise RuntimeError(\'Compiled harus menghasilkan 461 peserta WITH_ILS unik.\')\n    source[\'userid\']=source.userid.astype(int);source[\'courseid\']=source.courseid.astype(int)\n    original=files.frame(paths[2]);original[\'userid\']=original.userid.astype(int);original[\'courseid\']=original.courseid.astype(int)\n    if len(original)!=461 or original.userid.duplicated().any() or not set(original.split)=={\'train\',\'test\'} or original.split.eq(\'train\').sum()!=322 or original.split.eq(\'test\').sum()!=139:\n        raise RuntimeError(\'Keanggotaan praproses Pendikom final harus461peserta unik dengan322/139 split sumber.\')\n    source=source.set_index(\'userid\',drop=False).loc[original.userid].reset_index(drop=True)\n    if not np.array_equal(source[[\'userid\',\'courseid\']],original[[\'userid\',\'courseid\']]):raise RuntimeError(\'Identitas raw dan hasil praproses Pendikom berbeda.\')\n    original_ids=original.loc[original.split.eq(\'train\'),\'userid\'];p.stay_refs=fit_refs(p,source.loc[source.userid.isin(original_ids)])\n    reconstructed=p.build_pendikom_numeric(source);category=p.categorize_pendikom(reconstructed)\n    keep=[c for c in p.PENDIKOM_FEATURES if category.loc[source.userid.isin(original_ids),c].nunique(dropna=False)>1]\n    final_features=list(original.columns[3:]);final_cat=files.frame(paths[3]).set_index(\'userid\').loc[original.userid]\n    if keep!=final_features or not np.allclose(reconstructed[keep],original[keep],rtol=1e-12,atol=1e-10) or not np.array_equal(category[keep],final_cat[keep]):\n        raise RuntimeError(\'Raw compiled tidak merekonstruksi hasil praproses final. Jangan memakai dataset pengganti.\')\n    ref=files.frame(REF,True);ref[\'userid\']=ref.userid.astype(int);ref[\'courseid\']=ref.courseid.astype(int)\n    ref=ref.set_index(\'userid\',drop=False).loc[original.userid].reset_index(drop=True)\n    for c in p.ILS_CODE_COLS+[\'ils_ls_class\']:\n        if ref[c].astype(str).tolist()!=source[c].astype(str).tolist():raise RuntimeError(\'Label ILS raw dan LIB berbeda: \'+c)\n    truth=ils_codes(s,ref);source[\'identity\']=source.userid.astype(str);source[\'course\']=source.courseid.astype(str)\n    rawmeta=raw.loc[pd.to_numeric(raw.has_ils,errors=\'coerce\').eq(1)].copy();rawmeta[\'userid\']=pd.to_numeric(rawmeta.userid,errors=\'raise\').astype(int)\n    rawmeta=rawmeta.set_index(\'userid\').loc[original.userid]\n    for c in [\'first_activity\',\'last_activity\',\'ils_submitted\']:\n        source[c+\'_utc\']=timestamps(rawmeta[c]).to_numpy() if c in rawmeta else np.nan\n    for c in ref:\n        if c.startswith(\'ils_\'):source[c]=ref[c].to_numpy()\n    sources={\'LMS\':main,\'PENDIKOM\':source};rules={\n        \'LMS\':pd.DataFrame(s.expected_rules(\'LMS\'),columns=[\'feature\',\'dimension\',\'low_supports\',\'high_supports\']),\n        \'PENDIKOM\':f.pendikom_semantic_rules()}\n    audit=[]\n    for cohort,df in sources.items():\n        audit.append({\'cohort\':cohort,\'n_learners\':len(df),\'n_courses\':df.course.nunique(),\'minimum_course_n\':int(df.course.value_counts().min()),\n            \'median_course_n\':float(df.course.value_counts().median()),\'courses_lt5\':int(df.course.value_counts().lt(5).sum()),\'independent_ILS\':cohort==\'PENDIKOM\'})\n    u.save_csv(out/\'AUDIT/03_kohort_dan_kursus.csv\',pd.DataFrame(audit));u.save_json(out/\'AUDIT/04_verifikasi_rekonstruksi_sumber.json\',{\n        \'verified\':True,\'source_records\':len(raw),\'ILS_learners\':461,\'original_train_ids\':len(original_ids),\'original_feature_mask\':keep,\n        \'max_numeric_absolute_difference\':float(np.max(np.abs(reconstructed[keep].to_numpy()-original[keep].to_numpy()))),\n        \'ILS_raw_matches_LIB\':True,\'selection_scope\':\'descriptive observed engagement/course mix; not proof missing ILS is random\'})\n    u.save_json(out/\'PROTOKOL/02_sumber_fungsi.json\',pins)\n    u.unchanged(files,watched,[])\n    return {\'sources\':sources,\'rules\':rules,\'truth\':truth,\'watched\':watched,\'pins\':pins}\n\ndef split_plan(u,out,data,cfg):\n    from sklearn.model_selection import GroupKFold\n    contexts=[];audit=[]\n    for cohort in cfg[\'cohorts\']:\n        df=data[\'sources\'][cohort]\n        for fold,(train,test) in enumerate(GroupKFold(cfg[\'n_folds\']).split(df,groups=df.course),1):\n            contexts.append({\'cohort\':cohort,\'protocol\':\'COURSE_GROUP\',\'fold\':fold,\'train\':train,\'test\':test,\'excluded\':np.array([],int)})\n        if cohort==\'LMS\':\n            audit.append({\'cohort\':cohort,\'protocol\':\'TEMPORAL\',\'status\':\'AUDIT_ONLY\',\'reason\':\'Existing numeric reconstruction uses full-course P95 observation windows; no forward-safe event-level reconstruction is claimed.\'})\n            continue\n        if not cfg[\'temporal_pendikom\']:\n            audit.append({\'cohort\':cohort,\'protocol\':\'TEMPORAL\',\'status\':\'NOT_REQUESTED\'});continue\n        first,last,survey=[df[c+\'_utc\'].to_numpy(float) for c in [\'first_activity\',\'last_activity\',\'ils_submitted\']]\n        cut=pd.Timestamp(cfg[\'temporal_cutoff_utc\']).timestamp();valid=np.isfinite(first)&np.isfinite(last)&np.isfinite(survey)&(first<=last)&(first<=survey)\n        train=np.flatnonzero(valid&(last<cut));test=np.flatnonzero(valid&(first>=cut));overlap=set(df.iloc[train].course)&set(df.iloc[test].course)\n        purged=train[df.iloc[train].course.isin(overlap).to_numpy()];train=train[~df.iloc[train].course.isin(overlap).to_numpy()]\n        excluded=np.setdiff1d(np.arange(len(df)),np.r_[train,test]);eligible=len(train)>=100 and len(test)>=30 and df.iloc[train].course.nunique()>=2 and df.iloc[test].course.nunique()>=2\n        row={\'cohort\':cohort,\'protocol\':\'TEMPORAL\',\'status\':\'ELIGIBLE\' if eligible else \'INELIGIBLE\',\'cutoff_utc\':cfg[\'temporal_cutoff_utc\'],\n            \'train_n\':len(train),\'test_n\':len(test),\'excluded_n\':len(excluded),\'invalid_timestamp_n\':int((~valid).sum()),\'purged_training_course_overlap_n\':len(purged),\n            \'reason\':\'Fixed calendar cutoff; whole observation windows; no score-based cutoff search; no early-prediction claim.\'}\n        audit.append(row)\n        x=df[[\'identity\',\'course\']].copy();x[\'role\']=\'EXCLUDED\';x.loc[train,\'role\']=\'TRAIN\';x.loc[test,\'role\']=\'TEST\'\n        x[\'first_utc\']=pd.to_datetime(first,unit=\'s\',utc=True);x[\'last_utc\']=pd.to_datetime(last,unit=\'s\',utc=True);x[\'ILS_utc\']=pd.to_datetime(survey,unit=\'s\',utc=True)\n        u.save_csv(out/\'AUDIT/05_temporal_eligibility_pendikom.csv\',x)\n        if eligible:contexts.append({\'cohort\':cohort,\'protocol\':\'TEMPORAL\',\'fold\':1,\'train\':train,\'test\':test,\'excluded\':excluded})\n    u.save_csv(out/\'PROTOKOL/03_kelayakan_waktu.csv\',pd.DataFrame(audit))\n    for c in contexts:\n        df=data[\'sources\'][c[\'cohort\']];a=df.iloc[c[\'train\']];b=df.iloc[c[\'test\']]\n        if set(a.identity)&set(b.identity) or set(a.course)&set(b.course):raise RuntimeError(\'Split32 memiliki tumpang tindih peserta/kursus.\')\n        c[\'key\']=c[\'cohort\']+\'/\'+c[\'protocol\']+(\'/F%02d\'%c[\'fold\'])\n    return contexts,audit\n\ndef transform_context(u,out,data,c,modules):\n    s,p,l=modules[\'SEM\'],modules[\'PEND\'],modules[\'LMS\'];df=data[\'sources\'][c[\'cohort\']]\n    tr=df.iloc[c[\'train\']].copy().reset_index(drop=True);te=df.iloc[c[\'test\']].copy().reset_index(drop=True);folder=out/\'SPLIT\'/c[\'key\']\n    if c[\'cohort\']==\'LMS\':\n        a=tr.rename(columns={\'identity\':\'user\'});b=te.rename(columns={\'identity\':\'user\'});state,thresholds=l.fit_main_categorizer(a)\n        x=l.apply_main_categorizer(a,state);y=l.apply_main_categorizer(b,state);features=s.MAIN\n        u.save_csv(folder/\'03_thresholds_train.csv\',thresholds);u.save_json(folder/\'04_train_state.json\',state)\n    else:\n        p.stay_refs=fit_refs(p,tr);a=p.build_pendikom_numeric(tr);b=p.build_pendikom_numeric(te);x=p.categorize_pendikom(a);y=p.categorize_pendikom(b)\n        features=[f for f in p.PENDIKOM_FEATURES if x[f].nunique(dropna=False)>1]\n        u.save_json(folder/\'04_train_state.json\',{\'fit_train_ids\':tr.identity.tolist(),\'duration_references\':{\n            k:{\'by_training_course\':{str(int(i)):float(v) for i,v in pair[0].items()},\'global_train_q75\':float(pair[1])} for k,pair in p.stay_refs.items()},\n            \'unseen_course_policy\':\'use global positive Q75 from this training fold, never test-course statistics\'})\n        u.save_csv(folder/\'03_fixed_graf_thresholds.csv\',pd.DataFrame([{\'feature\':f,\'low\':p.PENDIKOM_THRESHOLDS[f][0],\'high\':p.PENDIKOM_THRESHOLDS[f][1]} for f in p.PENDIKOM_FEATURES]))\n        u.save_csv(folder/\'05_feature_mask_train.csv\',pd.DataFrame([{\'feature\':f,\'n_unique_train\':x[f].nunique(dropna=False),\'used\':f in features} for f in p.PENDIKOM_FEATURES]))\n        for name,frame,original in [(\'01_train_numeric.csv\',a,tr),(\'02_test_numeric.csv\',b,te)]:\n            frame=frame[[\'userid\',\'courseid\']+p.PENDIKOM_FEATURES];u.save_csv(folder/name,frame)\n    rules=data[\'rules\'][c[\'cohort\']];rules=rules.loc[rules.feature.isin(features)].copy()\n    if set(rules.dimension)!=set(s.DIMS):raise RuntimeError(\'Tidak semua dimensi memiliki evidence pada fitur train: \'+c[\'key\'])\n    X=x[features].astype(int).astype(str).to_numpy(object);Y=y[features].astype(int).astype(str).to_numpy(object)\n    if not np.isin(X,[\'1\',\'2\',\'3\']).all() or not np.isin(Y,[\'1\',\'2\',\'3\']).all():raise RuntimeError(\'Kategori di luar1/2/3.\')\n    for name,frame,raw in [(\'01_train_categorical.csv\',x,tr),(\'02_test_categorical.csv\',y,te)]:\n        saved=pd.DataFrame({\'identity\':raw.identity,\'course\':raw.course,\'source_row_index\':c[\'train\'] if \'train\' in name else c[\'test\']})\n        for f in features:saved[f]=frame[f].to_numpy(int)\n        u.save_csv(folder/name,saved)\n    u.save_csv(folder/\'06_semantic_rules.csv\',rules)\n    train_identity={\'ids\':tr.identity.tolist(),\'courses\':tr.course.tolist(),\'features\':features,\'X\':X.tolist(),\'rules\':rules.to_dict(\'records\')}\n    test_identity={\'ids\':te.identity.tolist(),\'courses\':te.course.tolist(),\'features\':features,\'X\':Y.tolist()}\n    ctx={**c,\'X\':X,\'Y\':Y,\'features\':features,\'rules\':rules,\'tr\':tr,\'te\':te,\'train_hash\':u.digest(train_identity),\'test_hash\':u.digest(test_identity)}\n    u.save_json(folder/\'07_split_audit.json\',{\'key\':c[\'key\'],\'train_n\':len(tr),\'test_n\':len(te),\'train_courses\':tr.course.nunique(),\'test_courses\':te.course.nunique(),\n        \'participant_overlap\':0,\'course_overlap\':0,\'train_matrix_sha256\':ctx[\'train_hash\'],\'test_matrix_sha256\':ctx[\'test_hash\'],\'metadata_not_used_as_predictors\':[\'identity\',\'course\',\'split\',\'timestamps\',\'ILS labels\']})\n    return ctx\n\ndef archive_record(u,files,out,cp,reason):\n    old=u.read_checkpoint(cp)\n    if not old:return\n    dest=out/\'KONTROL/HISTORY\'/uuid.uuid4().hex[:16];copied=[]\n    for i,(relative,want) in enumerate(old.get(\'outputs\',{}).items()):\n        path=files.path(relative)\n        if not path.resolve().is_relative_to(out.resolve()):raise RuntimeError(\'Artefak cache bukan milik tahap32.\')\n        if path.is_file():\n            data=path.read_bytes();target=dest/\'ARTIFACTS\'/(\'%03d\'%i+\'\'.join(path.suffixes));u.atomic(target,data)\n            copied.append({\'original\':relative,\'archive\':str(target.relative_to(files.root)).replace(\'\\\\\',\'/\'),\'expected_sha256\':want,\'actual_sha256\':u.sha(data)})\n    u.save_json(dest/\'record.json\',{\'reason\':reason,\'previous_checkpoint\':str(cp.relative_to(files.root)).replace(\'\\\\\',\'/\'),\'previous_record\':old,\'artifacts\':copied,\'archived_utc\':utc()})\n\ndef checkpoint_begin(u,files,out,cp,signature,force):\n    cached=u.cache_read(files,cp,signature,force)\n    if cached:return cached\n    archive_record(u,files,out,cp,\'force\' if force else \'invalid_or_interrupted_cache\')\n    u.save_json(cp,{\'status\':\'berjalan\',\'signature\':signature,\'started_utc\':utc()});return None\n\ndef seed_for(ctx,run):return 161600+(100000 if ctx[\'cohort\']==\'PENDIKOM\' else 0)+(10000 if ctx[\'protocol\']==\'TEMPORAL\' else 0)+ctx[\'fold\']*100+run\n\ndef profile_codes(s,profile):return s.numeric_profiles(profile).astype(np.int8)\n\ndef empty_profiles(s,ids):\n    frame=pd.DataFrame({\'identity\':ids,\'cluster_id\':-1,\'ls_class\':\'AMBIGUOUS\',\'mapping_status\':\'UNRESOLVED\'})\n    for key in s.KEYS:frame[key]=\'Balanced\';frame[key+\'_score\']=np.nan\n    return frame\n\ndef fit_job(u,files,out,ctx,method,run,modules,code,cfg):\n    core,s,bk=modules[\'CORE\'],modules[\'SEM\'],modules[\'BK\'];seed=seed_for(ctx,run)\n    scientific={\'context\':ctx[\'key\'],\'train_sha256\':ctx[\'train_hash\'],\'method\':method,\'run\':run,\'seed\':seed,\'k\':16,\'max_iter\':100,\'CARE\':CARE}\n    sig=u.digest({\'code\':code,\'scientific\':scientific});key=sig[:24];folder=out/\'BANK\'/key;cp=out/\'KONTROL/FITS\'/(key+\'.json\')\n    record=checkpoint_begin(u,files,out,cp,sig,cfg[\'force\'])\n    if record:return record,\'CACHE\'\n    row={**scientific,\'cohort\':ctx[\'cohort\'],\'protocol\':ctx[\'protocol\'],\'fold\':ctx[\'fold\'],\'train_n\':len(ctx[\'X\']),\'features\':len(ctx[\'features\']),\n        \'outcome\':\'STARTED\',\'exception_type\':\'\',\'exception_message\':\'\',\'started_utc\':utc()};outputs=[]\n    try:\n        start=time.perf_counter()\n        try:\n            if len({tuple(x) for x in ctx[\'X\']})<16:raise ValueError(\'Fewer than16 unique train vectors.\')\n            detail=None\n            if method==\'BK-Modes\':detail=bk.bk_modes_init(ctx[\'X\'],16,max_iter=100,return_details=True);init=detail[\'modes\']\n            else:init=core.init_by_name(method,ctx[\'X\'],16,seed=seed,care_config=CARE)\n            fit=core.kmodes_fit(ctx[\'X\'],init,max_iter=100);labels=np.asarray(fit[\'labels\'],int);modes=np.asarray(fit[\'modes\'],object)\n            count=np.bincount(labels,minlength=16);nearest,distance=core.assign_nearest(ctx[\'X\'],modes)\n            initial=float(core.initial_cost(ctx[\'X\'],init));final=float(fit[\'cost\'])\n            if modes.shape!=(16,len(ctx[\'features\'])) or (count<=0).any() or final>initial+1e-12 or int(fit[\'iterations\'])<1 or not np.array_equal(labels,nearest) or final!=float(distance.sum()):raise ValueError(\'Invalid partition/cost/empty cluster.\')\n            mapping,evidence=s.interpret(ctx[\'X\'],modes,labels,ctx[\'features\'],ctx[\'rules\'])\n            target=s.profiles(mapping,labels,ctx[\'tr\'].identity.tolist(),ctx[\'train\'])\n            buf=io.BytesIO();np.savez_compressed(buf,initial_modes=np.asarray(init,str),modes=np.asarray(modes,str),labels=labels)\n            path=folder/\'01_partition.npz\';u.atomic(path,buf.getvalue());outputs.append(path)\n            for filename,frame in [(\'02_cluster_profiles.csv\',mapping),(\'03_evidence.csv\',evidence),(\'04_train_targets.csv.gz\',target)]:\n                path=folder/filename;u.save_csv(path,frame);outputs.append(path)\n            if detail is not None:path=folder/\'05_BK_trace.json\';u.save_json(path,detail[\'split_trace\']);outputs.append(path)\n            row.update(outcome=\'OK\',initial_cost=initial,final_cost=final,iterations=int(fit[\'iterations\']),min_cluster=int(count.min()),max_cluster=int(count.max()),\n                unique_initial_modes=len({tuple(x) for x in init}),iteration_cap=int(fit[\'iterations\'])>=100,n_unresolved_train=int(target.mapping_status.ne(\'RESOLVED\').sum()))\n        except Exception as error:\n            row.update(outcome=\'RESOURCE_ERROR\' if isinstance(error,MemoryError) else \'FIT_OR_MAPPING_ERROR\',exception_type=type(error).__name__,exception_message=str(error))\n        row.update(elapsed_fit_and_audit_sec=time.perf_counter()-start,finished_utc=utc())\n        path=folder/\'00_fit_audit.json\';u.save_json(path,row);outputs.append(path)\n        return u.cache_write(files,cp,sig,outputs,{\'row\':row,\'scientific\':scientific}), \'FIT\'\n    except BaseException as error:\n        u.save_json(cp,{\'status\':\'terinterupsi\',\'signature\':sig,\'scientific\':scientific,\'error\':type(error).__name__+\': \'+str(error),\'finished_utc\':utc()});raise\n\ndef artifact(record,name):\n    matches=[p for p in record[\'outputs\'] if p.endswith(\'/\'+name)]\n    if len(matches)!=1:raise RuntimeError(\'Artefak cache tidak lengkap: \'+name)\n    return matches[0]\n\ndef predictions_from_fit(u,files,out,ctx,record,modules,code,cfg):\n    core,s,prof=modules[\'CORE\'],modules[\'SEM\'],modules[\'PROF\'];r=record[\'row\'];outputs=[]\n    sig=u.digest({\'code\':code,\'fit_record\':record_digest(u,record),\'test_sha256\':ctx[\'test_hash\']});cp=out/\'KONTROL/PREDICTIONS\'/(sig[:24]+\'.json\');folder=out/\'PREDIKSI/BANK\'/sig[:24]\n    cached=checkpoint_begin(u,files,out,cp,sig,cfg[\'force\'])\n    if cached:\n        return {name:files.frame(path,True) for name,path in cached[\'profiles\'].items()},cached\n    profile={};ids=ctx[\'te\'].identity.tolist()\n    if r[\'outcome\']!=\'OK\':\n        profile[r[\'method\']+\'_DIRECT\']=empty_profiles(s,ids)\n        if r[\'method\']==\'CARE-KM\':\n            for name in [\'CARE_MEMBER_ONLY\',\'CARE_MEMBER_FIRST\']:profile[name]=empty_profiles(s,ids)\n    else:\n        with np.load(io.BytesIO(files.read(artifact(record,\'01_partition.npz\'))),allow_pickle=False) as z:modes=z[\'modes\'].astype(object)\n        mapping=files.frame(artifact(record,\'02_cluster_profiles.csv\'));labels,distance=core.assign_nearest(ctx[\'Y\'],modes)\n        profile[r[\'method\']+\'_DIRECT\']=s.profiles(mapping,labels,ids,ctx[\'test\'])\n        assignment=pd.DataFrame({\'identity\':ids,\'course\':ctx[\'te\'].course,\'source_row_index\':ctx[\'test\'],\'cluster_id\':labels,\'SMD_to_nearest_mode\':distance})\n        path=folder/\'00_test_assignments.csv.gz\';u.save_csv(path,assignment);outputs.append(path)\n        if r[\'method\']==\'CARE-KM\':\n            evidence=files.frame(artifact(record,\'03_evidence.csv\'))\n            for name,operator in [(\'CARE_MEMBER_ONLY\',prof.MEMBER),(\'CARE_MEMBER_FIRST\',prof.CONSISTENT)]:\n                alt,decisions=prof.make_mapping(s,mapping,evidence,operator)\n                profile[name]=s.profiles(alt,labels,ids,ctx[\'test\'])\n                for suffix,frame in [(\'mapping\',alt),(\'decisions\',decisions)]:\n                    path=folder/(name+\'_\'+suffix+\'.csv\');u.save_csv(path,frame);outputs.append(path)\n    paths={}\n    for name,frame in profile.items():\n        path=folder/(name+\'.csv.gz\');u.save_csv(path,frame);outputs.append(path);paths[name]=str(path.relative_to(files.root)).replace(\'\\\\\',\'/\')\n    cached=u.cache_write(files,cp,sig,outputs,{\'profiles\':paths,\'fit_outcome\':r[\'outcome\'],\'test_sha256\':ctx[\'test_hash\']})\n    return profile,cached\n\ndef direct_rule_job(u,files,out,ctx,modules,code,cfg):\n    s,p=modules[\'SEM\'],modules[\'PROF\'];sig=u.digest({\'code\':code,\'train\':ctx[\'train_hash\'],\'test\':ctx[\'test_hash\'],\'rule\':\'DIRECT_BEHAVIORAL_RULES\'})\n    cp=out/\'KONTROL/DIRECT\'/(sig[:24]+\'.json\');path=out/\'PREDIKSI/DIRECT\'/(sig[:24]+\'.csv.gz\')\n    cached=checkpoint_begin(u,files,out,cp,sig,cfg[\'force\'])\n    if cached:return files.frame(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'),True),cached\n    pred,e=p.direct_profiles(s,ctx[\'Y\'],ctx[\'te\'].identity.tolist(),ctx[\'features\'],ctx[\'rules\'],ctx[\'test\'])\n    ep=path.with_name(path.stem+\'.evidence.csv.gz\');u.save_csv(path,pred);u.save_csv(ep,e)\n    return pred,u.cache_write(files,cp,sig,[path,ep],{})\n\ndef train_model_job(u,files,out,ctx,record,family,modules,code,cfg):\n    import joblib\n    from sklearn.dummy import DummyClassifier\n    s,f=modules[\'SEM\'],modules[\'FACTORY\'];r=record[\'row\'];outputs=[]\n    if r[\'outcome\']==\'OK\':target=files.frame(artifact(record,\'04_train_targets.csv.gz\'),True)\n    else:target=empty_profiles(s,ctx[\'tr\'].identity.tolist())\n    if target.identity.tolist()!=ctx[\'tr\'].identity.tolist():raise RuntimeError(\'Target train classifier tidak sejajar.\')\n    y=profile_codes(s,target);matrix=ctx[\'X\'].astype(int);is_main=ctx[\'cohort\']==\'LMS\'\n    def factories():return f.build_main_classifiers() if is_main else f.build_external_classifiers()\n    signature=u.digest({\'code\':code,\'train_sha256\':ctx[\'train_hash\'],\'targets\':target[[\'identity\']+s.KEYS+[\'ls_class\']].to_dict(\'records\'),\'family\':family,\'model_protocol\':\'original22-zeroILS-train\'})\n    cp=out/\'KONTROL/MODELS\'/(signature[:24]+\'.json\');folder=out/\'MODELS\'/signature[:24]\n    cached=checkpoint_begin(u,files,out,cp,signature,cfg[\'force\'])\n    if cached:return joblib.load(files.path(artifact(cached,\'model.joblib\'))),cached,\'CACHE\'\n    models=[];audit=[]\n    for di in range(4 if is_main else 1):\n        labels=y[:,di] if is_main else target.ls_class.to_numpy();valid=(y[:,di]!=0) if is_main else (y!=0).all(axis=1)\n        used=np.flatnonzero(valid);unique=np.unique(labels[valid]);model=None;outcome=\'OK\';error=\'\'\n        try:\n            if len(unique)==1:model=DummyClassifier(strategy=\'constant\',constant=unique[0]);outcome=\'SINGLE_CLASS_CONSTANT\'\n            elif len(unique)>1:model=factories()[family]\n            else:outcome=\'NO_RESOLVED_TRAIN_TARGETS\'\n            if model is not None:model.fit(matrix[used],labels[used])\n        except Exception as exc:model=None;outcome=\'MODEL_FIT_ERROR\';error=type(exc).__name__+\': \'+str(exc)\n        models.append(model);audit.append({\'family\':family,\'dimension\':s.KEYS[di] if is_main else \'Base_LS\',\'n_train_total\':len(matrix),\'n_train_used\':len(used),\n            \'excluded_unresolved_train\':int((~valid).sum()),\'n_classes\':len(unique),\'outcome\':outcome,\'error\':error,\'ILS_train_targets\':0,\n            \'training_ids_sha256\':u.digest(ctx[\'tr\'].identity.iloc[used].tolist())})\n    bundle={\'models\':models,\'audit\':audit,\'features\':ctx[\'features\'],\'cohort\':ctx[\'cohort\'],\'train_sha256\':ctx[\'train_hash\'],\'family\':family,\n        \'train_target\':\'CARE-KM mode-first CDS; never ILS\',\'runtime\':code[\'runtime\']}\n    buf=io.BytesIO();joblib.dump(bundle,buf);path=folder/\'model.joblib\';u.atomic(path,buf.getvalue());outputs.append(path)\n    path=folder/\'fit_audit.csv\';u.save_csv(path,pd.DataFrame(audit));outputs.append(path)\n    record=u.cache_write(files,cp,signature,outputs,{\'audit\':audit,\'train_sha256\':ctx[\'train_hash\']});return bundle,record,\'MODEL_FIT\'\n\ndef model_predict_job(u,files,out,ctx,bundle,record,modules,code,cfg):\n    s=modules[\'SEM\'];sig=u.digest({\'code\':code,\'model_record\':record_digest(u,record),\'test\':ctx[\'test_hash\']});cp=out/\'KONTROL/MODEL_PREDICTIONS\'/(sig[:24]+\'.json\')\n    path=out/\'PREDIKSI/MODELS\'/(sig[:24]+\'.csv.gz\');cached=checkpoint_begin(u,files,out,cp,sig,cfg[\'force\'])\n    if cached:return files.frame(str(path.relative_to(files.root)).replace(\'\\\\\',\'/\'),True),cached\n    matrix=ctx[\'Y\'].astype(int);codes=np.zeros((len(matrix),4),np.int8);errors=[]\n    for di,model in enumerate(bundle[\'models\']):\n        if model is None:continue\n        try:\n            pred=model.predict(matrix)\n            if ctx[\'cohort\']==\'LMS\':codes[:,di]=np.asarray(pred,int)\n            else:\n                for i,label in enumerate(pred):\n                    poles=next((combo for combo,value in s.LS_LOOKUP.items() if value==str(label)),None)\n                    if poles is None:raise ValueError(\'Unknown Base LS class.\')\n                    codes[i]=[1 if pole==p[0] else -1 for pole,p in zip(poles,s.POLES)]\n        except Exception as error:errors.append({\'dimension\':di,\'error\':type(error).__name__+\': \'+str(error)})\n    pred=empty_profiles(s,ctx[\'te\'].identity.tolist())\n    for di,(key,poles) in enumerate(zip(s.KEYS,s.POLES)):pred[key]=np.where(codes[:,di]==1,s.ABBR[poles[0]],np.where(codes[:,di]==-1,s.ABBR[poles[1]],\'Balanced\'))\n    resolved=(codes!=0).all(axis=1);pred[\'mapping_status\']=np.where(resolved,\'RESOLVED\',\'UNRESOLVED\')\n    pred[\'ls_class\']=[s.LS_LOOKUP[tuple(p[0] if v==1 else p[1] for v,p in zip(row,s.POLES))] if ok else \'AMBIGUOUS\' for row,ok in zip(codes,resolved)]\n    u.save_csv(path,pred);return pred,u.cache_write(files,cp,sig,[path],{\'errors\':errors})\n\ndef prediction_rows(s,ctx,profile,pathway,run,seed,outcome=\'OK\'):\n    if profile.identity.astype(str).tolist()!=ctx[\'te\'].identity.tolist():raise RuntimeError(\'Prediksi tidak sejajar dengan seluruh peserta test.\')\n    frame=pd.DataFrame({\'cohort\':ctx[\'cohort\'],\'protocol\':ctx[\'protocol\'],\'fold\':ctx[\'fold\'],\'identity\':ctx[\'te\'].identity.tolist(),\n        \'course\':ctx[\'te\'].course.tolist(),\'pathway\':pathway,\'run\':run,\'seed\':seed,\'outcome\':outcome})\n    for di,key in enumerate(s.KEYS):frame[key]=profile_codes(s,profile)[:,di]\n    frame[\'profile_resolved\']=(frame[s.KEYS].to_numpy()!=0).all(axis=1);frame[\'ls_class\']=profile.ls_class.to_numpy()\n    return frame\n\ndef record_digest(u,record):return u.digest({k:v for k,v in record.items() if k!=\'record_sha256\'})\n\ndef interval(values,alpha=.05):\n    values=np.asarray(values,float);values=values[np.isfinite(values)]\n    if len(values)<100:return (np.nan,np.nan,len(values))\n    lo,hi=np.quantile(values,[alpha/2,1-alpha/2]);return float(lo),float(hi),len(values)\n\ndef bootstrap_weights(u,out,cohort,protocol,courses,ids,cfg):\n    groups,inverse=np.unique(np.asarray(courses,str),return_inverse=True);g=len(groups)\n    rng=np.random.default_rng(cfg[\'bootstrap_seed\']+(100 if cohort==\'LMS\' else 0)+(1 if protocol==\'TEMPORAL\' else 0))\n    weights=rng.multinomial(g,np.full(g,1/g),size=cfg[\'bootstrap_repetitions\']).astype(np.uint16)\n    folder=out/\'STATISTIK\'/cohort/protocol;buf=io.BytesIO()\n    np.savez_compressed(buf,course_ids=groups,learner_ids=np.asarray(ids,str),learner_course_index=inverse,weights=weights)\n    path=folder/\'01_course_bootstrap_weights.npz\';u.atomic(path,buf.getvalue())\n    return weights,inverse,groups,path\n\ndef evaluate_scheme(u,files,out,frame,data,s,code,cfg):\n    cohort=frame.cohort.iloc[0];protocol=frame.protocol.iloc[0];folder=out/\'EVALUASI\'/cohort/protocol\n    reference_fields=[\'identity\',\'course\']+[\'ils_\'+key+\'_code\' for key in s.KEYS]+[\'ils_ls_class\']\n    reference_sha=u.digest(data[\'sources\'][cohort][reference_fields].to_dict(\'records\')) if cohort==\'PENDIKOM\' else \'teacher_is_in_prediction_rows\'\n    sig=u.digest({\'code\':code,\'prediction_rows\':u.sha(frame.to_csv(index=False,lineterminator=\'\\n\').encode()),\'reference_sha256\':reference_sha,\n        \'bootstrap\':{k:cfg[k] for k in [\'bootstrap_repetitions\',\'bootstrap_seed\']}})\n    cp=out/\'KONTROL/EVALUATION\'/(cohort+\'_\'+protocol+\'.json\');cached=checkpoint_begin(u,files,out,cp,sig,cfg[\'force\'])\n    if cached:\n        return {key:files.frame(path) for key,path in cached[\'tables\'].items()},\'CACHE\'\n    spine=frame[[\'identity\',\'course\',\'fold\']].drop_duplicates().reset_index(drop=True)\n    if spine.identity.duplicated().any():raise RuntimeError(\'Peserta muncul pada beberapa fold test dalam satu protokol.\')\n    ids=spine.identity.tolist();n=len(ids);truth=None\n    if cohort==\'PENDIKOM\':\n        ref=data[\'sources\'][cohort].set_index(\'identity\').loc[ids].reset_index();truth=ils_codes(s,ref);reference=\'INDEPENDENT_ILS\'\n    else:\n        teacher=frame.loc[frame.pathway.eq(\'CARE-KM_DIRECT\')&frame.run.eq(0)].set_index(\'identity\').loc[ids]\n        truth=teacher[s.KEYS].to_numpy(int);reference=\'CARE_TRAIN_MAPPING_FIDELITY_NOT_ILS\'\n    weights,inverse,groups,wp=bootstrap_weights(u,out,cohort,protocol,spine.course,ids,cfg);outputs=[wp]\n    summaries=[];confusions=[];per_run=[];per_course=[];refcounts=[];boot={};points={};pathways=[]\n    for di,dim in enumerate(s.KEYS):\n        for fold in [\'ALL\']+list(sorted(spine.fold.unique())):\n            mask=np.ones(n,bool) if fold==\'ALL\' else spine.fold.eq(fold).to_numpy();t=truth[mask,di]\n            refcounts.append({\'cohort\':cohort,\'protocol\':protocol,\'fold\':fold,\'dimension\':dim,\'n_learners\':len(t),\'first_pole\':int((t==1).sum()),\'second_pole\':int((t==-1).sum()),\'unresolved_reference\':int((t==0).sum()),\'reference\':reference})\n    for pathway,g in frame.groupby(\'pathway\',sort=False):\n        runs=sorted(g.run.unique());pred=[]\n        for run in runs:\n            one=g.loc[g.run.eq(run)]\n            if len(one)!=n or one.identity.duplicated().any() or set(one.identity)!=set(ids):raise RuntimeError(\'Prediksi per run belum mencakup seluruh test: \'+pathway)\n            pred.append(one.set_index(\'identity\').loc[ids,s.KEYS].to_numpy(int))\n        prediction=np.stack(pred);masks=s.confusion_masks(truth,prediction);counts=masks.sum(axis=0);values=s.count_metrics(counts)\n        groupcounts=np.zeros((len(groups),len(runs),4,6));np.add.at(groupcounts,inverse,masks)\n        boot_counts=np.einsum(\'bg,grdc->brdc\',weights.astype(float),groupcounts,optimize=True)\n        draws=s.count_metrics(boot_counts);mean_draws=np.mean(draws,axis=1)\n        point=np.mean(values,axis=0);points[pathway]=np.mean(point,axis=0);boot[pathway]=np.mean(mean_draws,axis=1);pathways.append(pathway)\n        for ri,run in enumerate(runs):\n            for di,key in enumerate(s.KEYS):\n                a,b,c,d,e,f=counts[ri,di].astype(int)\n                confusions.append({\'cohort\':cohort,\'protocol\':protocol,\'pathway\':pathway,\'run\':run,\'dimension\':key,\n                    \'true_first_pred_first\':a,\'true_first_pred_second\':b,\'true_first_abstain\':c,\'true_second_pred_first\':d,\'true_second_pred_second\':e,\'true_second_abstain\':f})\n                per_run.append({\'cohort\':cohort,\'protocol\':protocol,\'pathway\':pathway,\'run\':run,\'dimension\':key,\n                    **dict(zip(METRICS,values[ri,di])),\'coverage\':float((prediction[ri,:,di]!=0).mean()),\'reference\':reference})\n        for di,key in enumerate(s.KEYS+[\'OVERALL\']):\n            value=point[di] if di<4 else np.mean(point,axis=0);sample=mean_draws[:,di,:] if di<4 else np.mean(mean_draws,axis=1)\n            row={\'cohort\':cohort,\'protocol\':protocol,\'pathway\':pathway,\'dimension\':key,\'n_test\':n,\'n_courses\':len(groups),\'n_runs\':len(runs),\'reference\':reference,\n                \'coverage\':float((prediction[:,:,di]!=0).mean()) if di<4 else float((prediction!=0).mean()),\n                \'full_profile_coverage\':float((prediction!=0).all(axis=2).mean()),\n                \'reference_coverage\':float((truth[:,di]!=0).mean()) if di<4 else float((truth!=0).mean())}\n            for mi,metric in enumerate(METRICS):\n                lo,hi,valid=interval(sample[:,mi]);row.update({metric:float(value[mi]),metric+\'_ci_low\':lo,metric+\'_ci_high\':hi,metric+\'_valid_bootstrap\':valid})\n                seed_values=values[:,di,mi] if di<4 else np.mean(values[:,:,mi],axis=1)\n                row[metric+\'_seed_sd\']=float(np.std(seed_values,ddof=1)) if len(seed_values)>1 else 0.\n            summaries.append(row)\n        for gi,course in enumerate(groups):\n            subset=inverse==gi;cv=s.count_metrics(groupcounts[gi]);pos=groupcounts[gi,:,:,0:3].sum(axis=2);neg=groupcounts[gi,:,:,3:6].sum(axis=2)\n            for di,key in enumerate(s.KEYS):\n                both=bool((pos[0,di]>0)&(neg[0,di]>0))\n                per_course.append({\'cohort\':cohort,\'protocol\':protocol,\'course\':course,\'pathway\':pathway,\'dimension\':key,\'n_test\':int(subset.sum()),\'both_reference_poles\':both,\n                    \'accuracy\':float(np.mean(cv[:,di,0])),\'balanced_accuracy\':float(np.mean(cv[:,di,1])) if both else np.nan,\n                    \'macro_f1\':float(np.mean(cv[:,di,2])) if both else np.nan,\'coverage\':float((prediction[:,subset,di]!=0).mean())})\n        buf=io.BytesIO();np.savez_compressed(buf,dimension_metrics=mean_draws,overall_metrics=boot[pathway],metric_names=np.asarray(METRICS),dimension_names=np.asarray(s.KEYS))\n        path=out/\'STATISTIK\'/cohort/protocol/(u.digest(pathway)[:16]+\'_bootstrap.npz\');u.atomic(path,buf.getvalue());outputs.append(path)\n    table=pd.DataFrame(summaries);course_table=pd.DataFrame(per_course);contrasts=[]\n    base=\'CARE-KM_DIRECT\';families={\'initializer\':[m+\'_DIRECT\' for m in s.METHODS if m!=\'CARE-KM\'],\n        \'profile_operator\':[\'BEHAVIORAL_DIRECT\',\'CARE_MEMBER_ONLY\',\'CARE_MEMBER_FIRST\'],\'classifier\':[\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\']}\n    for family,comparators in families.items():\n        available=[p for p in comparators if p in boot]\n        for other in available:\n            delta=boot[base]-boot[other]\n            for mi,metric in enumerate(METRICS):\n                lo,hi,nvalid=interval(delta[:,mi]);flo,fhi,_=interval(delta[:,mi],.05/len(available))\n                contrasts.append({\'cohort\':cohort,\'protocol\':protocol,\'family\':family,\'reference\':reference,\'comparison\':\'CARE_minus_\'+other,\'metric\':\'mean_\'+metric,\n                    \'point_difference\':float(points[base][mi]-points[other][mi]),\'ci_low\':lo,\'ci_high\':hi,\'valid_bootstrap\':nvalid,\n                    \'n_family_contrasts\':len(available),\'family95_ci_low\':flo if metric==\'balanced_accuracy\' else np.nan,\n                    \'family95_ci_high\':fhi if metric==\'balanced_accuracy\' else np.nan,\'multiplicity\':\'Bonferroni percentile family interval for primary mean BA; other CIs descriptive\'})\n    course_summary=course_table.groupby([\'cohort\',\'protocol\',\'pathway\',\'dimension\'],sort=False).agg(equal_course_mean_accuracy=(\'accuracy\',\'mean\'),n_courses=(\'course\',\'size\'),minimum_course_n=(\'n_test\',\'min\')).reset_index()\n    tables={\'summary\':table,\'confusions\':pd.DataFrame(confusions),\'per_run\':pd.DataFrame(per_run),\'reference_counts\':pd.DataFrame(refcounts),\n        \'per_course\':course_table,\'course_summary\':course_summary,\'paired\':pd.DataFrame(contrasts)};paths={}\n    for name,value in tables.items():\n        path=folder/(name+\'.csv\');u.save_csv(path,value);outputs.append(path);paths[name]=str(path.relative_to(files.root)).replace(\'\\\\\',\'/\')\n    u.save_json(folder/\'interpretation.json\',{\'reference\':reference,\'bootstrap_unit\':\'course, all learners/four dimensions/routes/seeds kept together\',\n        \'interval_scope\':\'conditional on these fitted folds and fixed runs; not full training uncertainty, not an independent-institution estimate\',\n        \'seeds\':\'metric per run, then average; no majority vote or best seed selection\',\'unresolved\':\'prediction0 is abstention/error; reference0 excluded only from reference-based metric with coverage reported\',\n        \'folds\':\'overlapping training sets; no independent-fold t-test/Friedman\',\'old70_30\':\'context only; different train fraction and references, no paired old-new effect claim\',\n        \'full_profile_coverage\':\'all4directions resolved; descriptive behavioral magnitudes are not ILS scores\',\'small_courses\':\'kept in all denominators; per-course BA/F1 unavailable when either reference pole absent\'})\n    outputs.append(folder/\'interpretation.json\');u.cache_write(files,cp,sig,outputs,{\'tables\':paths});return tables,\'EVALUATE\'\n\ndef figures(u,out,summary):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    paths=[]\n    for protocol in [\'COURSE_GROUP\',\'TEMPORAL\']:\n        x=summary.loc[summary.cohort.eq(\'PENDIKOM\')&summary.protocol.eq(protocol)&summary.dimension.eq(\'OVERALL\')].copy()\n        if x.empty:continue\n        x=x.set_index(\'pathway\');order=[n for n in [\'CARE-KM_DIRECT\',\'Cao_DIRECT\',\'Bai_DIRECT\',\'BK-Modes_DIRECT\',\'K-Modes++_DIRECT\',\'Random_DIRECT\',\'Huang_DIRECT\',\'HR-2023_DIRECT\',\n            \'BEHAVIORAL_DIRECT\',\'CARE_MEMBER_ONLY\',\'CARE_MEMBER_FIRST\',\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'] if n in x.index];x=x.loc[order]\n        fig,ax=plt.subplots(figsize=(9,7));y=np.arange(len(x));v=x.balanced_accuracy.to_numpy(float);lo=x.balanced_accuracy_ci_low.to_numpy(float);hi=x.balanced_accuracy_ci_high.to_numpy(float)\n        ax.hlines(y,lo,hi,color=\'#21618c\');ax.scatter(v,y,color=\'#21618c\',s=28)\n        ax.set_yticks(y,[n.replace(\'_DIRECT\',\' direct\').replace(\'CARE_MEMBER_ONLY\',\'CARE member-only\').replace(\'CARE_MEMBER_FIRST\',\'CARE member-first\') for n in x.index]);ax.invert_yaxis()\n        ax.set_xlabel(\'Mean balanced accuracy versus ILS (95% course-bootstrap CI)\');ax.set_xlim(0,1.03)\n        ax.set_title(\'Unseen courses\' if protocol==\'COURSE_GROUP\' else \'Later whole-record cohort (fixed cutoff)\');ax.grid(axis=\'x\',alpha=.2);fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:\n            path=out/\'GAMBAR\'/(\'01_ILS_\'+protocol+\'.\'+ext);path.parent.mkdir(parents=True,exist_ok=True);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n        plt.close(fig)\n    x=summary.loc[summary.cohort.eq(\'LMS\')&summary.protocol.eq(\'COURSE_GROUP\')&summary.dimension.eq(\'OVERALL\')&summary.pathway.isin([\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])]\n    if len(x):\n        fig,ax=plt.subplots(figsize=(7,4.8));v=x.balanced_accuracy.to_numpy(float);lo=x.balanced_accuracy_ci_low.to_numpy(float);hi=x.balanced_accuracy_ci_high.to_numpy(float)\n        ax.vlines(np.arange(len(x)),lo,hi,color=\'#21618c\');ax.scatter(np.arange(len(x)),v,color=\'#21618c\',s=30);ax.set_xticks(np.arange(len(x)),x.pathway.str.replace(\'CDS_\',\'\',regex=False))\n        ax.set_ylim(0,1.03);ax.set_ylabel(\'Mean balanced accuracy against frozen teacher\');ax.set_title(\'LMS classifier fidelity on unseen courses\');ax.text(.5,-.2,\'Teacher-derived references; not independent ILS validation\',transform=ax.transAxes,ha=\'center\',fontsize=11);fig.tight_layout()\n        for ext in [\'png\',\'pdf\']:\n            path=out/\'GAMBAR\'/(\'02_LMS_fidelity.\'+ext);path.parent.mkdir(parents=True,exist_ok=True);fig.savefig(path,dpi=300,bbox_inches=\'tight\');paths.append(path)\n        plt.close(fig)\n    return paths\n\ndef run(root,configuration,source,capsules):\n    from threadpoolctl import threadpool_limits\n    u=load_module(capsules[\'UTILS\'],\'_utils32\');files=u.Files(Path(root).expanduser().resolve());out=files.root/\'OUTPUT/TAMBAHAN\'/STAGE\n    out.mkdir(parents=True,exist_ok=True);control=out/\'KONTROL\';control.mkdir(parents=True,exist_ok=True)\n    fit_counts={\'FIT\':0,\'CACHE\':0};model_counts={\'MODEL_FIT\':0,\'CACHE\':0};records=[];model_rows=[];predictions=[];dependencies={}\n    try:\n        modules={k:load_module(v,\'_transport32_\'+k) for k,v in capsules.items() if isinstance(v,str) and k!=\'UTILS\'}\n        s=modules[\'SEM\'];cfg=config(configuration,s)\n        modules[\'CORE\']=load_module(files.read(\'LIB/care_km.py\').decode(\'utf-8-sig\'),\'_core32\',files.path(\'LIB/care_km.py\'))\n        own=u.notebook_hash(files,STAGE,\'transport32_configuration\',\'KONFIGURASI_32\');checks=[(STAGE,\'transport32_configuration\',\'KONFIGURASI_32\',own)]\n        code={\'implementation_sha256\':u.sha(source.encode()),\'capsules_sha256\':{k:u.sha(v.encode()) for k,v in capsules.items() if isinstance(v,str)},\n            \'notebook_code\':own,\'runtime\':u.runtime()}\n        u.save_json(out/\'PROTOKOL/01_konfigurasi.json\',{\'configuration\':cfg,\'code\':code,\'k\':16,\'CARE\':CARE,\'max_iter\':100,\'methods\':s.METHODS,\n            \'primary_endpoint\':\'Pendikom independent ILS mean balanced accuracy; LMS is teacher fidelity only\',\n            \'splits\':\'GroupKFold by course, no ILS stratification; default5fold about80/20, different from original70/30\',\n            \'temporal\':\'Pendikom fixed2021-07-01UTC, entire activity windows; crossing/invalid records excluded, training courses overlapping test purged; no score-based alternative cutoff\',\n            \'supervision\':\'ILS never used for clustering, semantic rules, threshold fitting, classifier targets or split choice\',\n            \'classifiers\':\'CARE mode-first only, exact stage22 NB/DT/SVM factories; original per-cohort representation and target formulation\',\n            \'label_missingness\':\'565vs461 observed engagement/course comparisons descriptive; missing ILS not assumed random\',\n            \'population\':\'same two selected cohorts, no new institution, no psychometric equivalence or instructional matching claim\',\n            \'LMS_representation\':\'course-disjoint evaluation conditional on existing full-course reconstruction; temporal LMS is audit-only\',\n            \'cache\':\'train-content fit keys; test predictions separate; contents/code/versions checked, failures retained; Colab output autosave has bounded retries\'})\n        data=prepare(u,files,out,capsules,modules,cfg);contexts,time_audit=split_plan(u,out,data,cfg)\n        plans=[]\n        for c in contexts:\n            df=data[\'sources\'][c[\'cohort\']];plans.append({\'context\':c[\'key\'],\'cohort\':c[\'cohort\'],\'protocol\':c[\'protocol\'],\'fold\':c[\'fold\'],\n                \'train_n\':len(c[\'train\']),\'test_n\':len(c[\'test\']),\'excluded_n\':len(c[\'excluded\']),\n                \'train_courses\':df.iloc[c[\'train\']].course.nunique(),\'test_courses\':df.iloc[c[\'test\']].course.nunique(),\'course_overlap\':0,\'learner_overlap\':0})\n        u.save_csv(out/\'PROTOKOL/04_rencana_split.csv\',pd.DataFrame(plans))\n        n_per_context=sum(cfg[\'stochastic_runs\'] if m in s.STOCHASTIC else 1 for m in s.METHODS);planned=len(contexts)*n_per_context\n        with threadpool_limits(limits=1):\n            for ci,c in enumerate(contexts,1):\n                print(\'Konteks\',ci,\'/\',len(contexts),c[\'key\'],flush=True)\n                u.unchanged(files,data[\'watched\'],checks);verify_sources(u,files,capsules);ctx=transform_context(u,out,data,c,modules)\n                context_frames=[];care_record=None\n                for method in s.ORDER:\n                    for run_index in range(cfg[\'stochastic_runs\'] if method in s.STOCHASTIC else 1):\n                        rec,origin=fit_job(u,files,out,ctx,method,run_index,modules,code,cfg);fit_counts[origin]+=1;records.append(rec[\'row\'])\n                        if method==\'CARE-KM\':care_record=rec\n                        profiles,pr=predictions_from_fit(u,files,out,ctx,rec,modules,code,cfg)\n                        for name,profile in profiles.items():context_frames.append(prediction_rows(s,ctx,profile,name,run_index,seed_for(ctx,run_index),rec[\'row\'][\'outcome\']))\n                        for path,sha in rec[\'outputs\'].items():dependencies[path]=sha\n                        for path,sha in pr[\'outputs\'].items():dependencies[path]=sha\n                        dependencies[str((control/\'FITS\'/(u.digest({\'code\':code,\'scientific\':rec[\'scientific\']})[:24]+\'.json\')).relative_to(files.root)).replace(\'\\\\\',\'/\')]=files.hash(str((control/\'FITS\'/(u.digest({\'code\':code,\'scientific\':rec[\'scientific\']})[:24]+\'.json\')).relative_to(files.root)).replace(\'\\\\\',\'/\'))\n                        u.save_json(control/\'status_eksekusi.json\',{\'status\':\'berjalan\',\'context\':c[\'key\'],\'completed_fit_jobs\':len(records),\'planned_fit_jobs\':planned,\'fit_counts\':fit_counts,\'model_counts\':model_counts})\n                    print(\'  \',method,\'selesai;\',fit_counts,flush=True)\n                direct,dr=direct_rule_job(u,files,out,ctx,modules,code,cfg);context_frames.append(prediction_rows(s,ctx,direct,\'BEHAVIORAL_DIRECT\',0,0))\n                dependencies.update(dr[\'outputs\'])\n                if cfg[\'classifiers\']:\n                    for family in [\'NB\',\'DT\',\'SVM\']:\n                        bundle,mr,origin=train_model_job(u,files,out,ctx,care_record,family,modules,code,cfg);model_counts[origin]+=1\n                        for row in bundle[\'audit\']:model_rows.append({\'cohort\':c[\'cohort\'],\'protocol\':c[\'protocol\'],\'fold\':c[\'fold\'],**row})\n                        pred,mp=model_predict_job(u,files,out,ctx,bundle,mr,modules,code,cfg)\n                        outcomes=[row[\'outcome\'] for row in bundle[\'audit\']];outcome=\'MODEL_FIT_ERROR\' if any(x in [\'MODEL_FIT_ERROR\',\'NO_RESOLVED_TRAIN_TARGETS\'] for x in outcomes) else \'PREDICTION_ERROR\' if mp.get(\'errors\') else \'OK\'\n                        context_frames.append(prediction_rows(s,ctx,pred,\'CDS_\'+family,0,161600,outcome));dependencies.update(mr[\'outputs\']);dependencies.update(mp[\'outputs\'])\n                context_pred=pd.concat(context_frames,ignore_index=True);path=out/\'PREDIKSI/PER_CONTEXT\'/(c[\'key\'].replace(\'/\',\'_\')+\'.csv.gz\');u.save_csv(path,context_pred)\n                predictions.append(context_pred);u.save_csv(out/\'AUDIT/06_fit_attempts.csv\',pd.DataFrame(records));u.save_csv(out/\'AUDIT/07_model_attempts.csv\',pd.DataFrame(model_rows))\n            full_pred=pd.concat(predictions,ignore_index=True);u.save_csv(out/\'PREDIKSI/01_semua_prediksi.csv.gz\',full_pred)\n            results=[];pairwise=[];refs=[];evaluations={}\n            for (cohort,protocol),frame in full_pred.groupby([\'cohort\',\'protocol\'],sort=False):\n                print(\'Evaluasi dan bootstrap kursus:\',cohort,protocol,flush=True)\n                tables,origin=evaluate_scheme(u,files,out,frame.reset_index(drop=True),data,s,code,cfg);results.append(tables[\'summary\']);pairwise.append(tables[\'paired\']);refs.append(tables[\'reference_counts\']);evaluations[cohort+\'/\'+protocol]=origin\n        summary=pd.concat(results,ignore_index=True);paired=pd.concat(pairwise,ignore_index=True);reference_counts=pd.concat(refs,ignore_index=True)\n        u.save_csv(out/\'TABEL/01_semua_hasil_per_dimensi.csv\',summary);u.save_csv(out/\'TABEL/02_perbandingan_berpasangan.csv\',paired);u.save_csv(out/\'TABEL/03_reference_counts.csv\',reference_counts)\n        u.save_csv(out/\'TABEL/04_Pendikom_ILS.csv\',summary.loc[summary.cohort.eq(\'PENDIKOM\')])\n        u.save_csv(out/\'TABEL/05_LMS_classifier_fidelity.csv\',summary.loc[summary.cohort.eq(\'LMS\')&summary.pathway.isin([\'CARE-KM_DIRECT\',\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])])\n        u.save_csv(out/\'TABEL/06_LMS_semantic_concordance_diagnostic.csv\',summary.loc[summary.cohort.eq(\'LMS\')&~summary.pathway.isin([\'CARE-KM_DIRECT\',\'CDS_NB\',\'CDS_DT\',\'CDS_SVM\'])])\n        figure_signature=u.digest({\'code\':code,\'summary\':u.sha((out/\'TABEL/01_semua_hasil_per_dimensi.csv\').read_bytes())})\n        if not u.cache_read(files,control/\'figures.json\',figure_signature,cfg[\'force\']):u.cache_write(files,control/\'figures.json\',figure_signature,figures(u,out,summary),{})\n        u.unchanged(files,data[\'watched\'],checks);verify_sources(u,files,capsules)\n        for path,want in dependencies.items():\n            if files.hash(path)!=want:raise RuntimeError(\'Artefak32 berubah selama eksekusi: \'+path)\n        u.save_json(out/\'PROTOKOL/05_manifest_masukan.json\',data[\'watched\']);u.save_json(out/\'PROTOKOL/06_manifest_artefak_fit_dan_prediksi.json\',dependencies)\n        failures=sum(r[\'outcome\']!=\'OK\' for r in records);model_failures=sum(r[\'outcome\'] in [\'MODEL_FIT_ERROR\',\'NO_RESOLVED_TRAIN_TARGETS\'] for r in model_rows)\n        temporal_ineligible=any(x.get(\'status\')==\'INELIGIBLE\' for x in time_audit)\n        prediction_failures=len(full_pred.loc[full_pred.outcome.eq(\'PREDICTION_ERROR\'),[\'cohort\',\'protocol\',\'fold\',\'pathway\']].drop_duplicates())\n        state=\'selesai_dengan_kegagalan_algoritmik\' if failures or model_failures or prediction_failures else \'selesai_kursus_temporal_tidak_layak\' if temporal_ineligible else \'selesai\'\n        full_course=set(cfg[\'cohorts\'])=={\'LMS\',\'PENDIKOM\'} and cfg[\'n_folds\']==5 and cfg[\'stochastic_runs\']==15 and cfg[\'bootstrap_repetitions\']>=10000 and cfg[\'classifiers\']\n        full_temporal=cfg[\'temporal_pendikom\'] and cfg[\'temporal_cutoff_utc\']==pd.Timestamp(DEFAULT[\'temporal_cutoff_utc\']).isoformat() and any(x.get(\'status\')==\'ELIGIBLE\' for x in time_audit)\n        status={\'status\':state,\'full_protocol\':bool(full_course and full_temporal and not temporal_ineligible),\'course_protocol_full\':bool(full_course),\'temporal_protocol_full\':bool(full_temporal),\n            \'completed_fit_jobs\':len(records),\'planned_fit_jobs\':planned,\'fit_counts\':fit_counts,\'failed_fit_jobs\':failures,\'model_counts\':model_counts,\'model_estimators\':len(model_rows),\'failed_model_estimators\':model_failures,\n            \'n_contexts\':len(contexts),\'n_prediction_rows\':len(full_pred),\'n_methods\':8,\'k\':16,\'q\':16,\'independent_ILS_training_targets\':0,\'failed_prediction_routes\':prediction_failures,\n            \'temporal_audit\':time_audit,\'temporal_protocol_requested\':cfg[\'temporal_pendikom\'],\'temporal_requested_completed\':not temporal_ineligible,\n            \'statistics\':evaluations,\'old_experiments_modified\':False,\'output\':str(out),\'finished_utc\':utc()}\n        u.save_json(control/\'manifest_keluaran.json\',{str(p.relative_to(files.root)).replace(\'\\\\\',\'/\'):u.sha(p.read_bytes()) for p in out.rglob(\'*\') if p.is_file() and control not in p.parents})\n        u.save_json(control/\'status_eksekusi.json\',status);print(\'Tahap32 selesai:\',status,flush=True)\n        return {\'summary\':summary,\'paired\':paired,\'status\':status,\'output\':str(out)}\n    except BaseException as error:\n        u.save_json(control/\'status_eksekusi.json\',{\'status\':\'masukan_belum_lengkap\' if isinstance(error,FileNotFoundError) else \'terhenti\',\'completed_fit_jobs\':len(records),\n            \'fit_counts\':fit_counts,\'model_counts\':model_counts,\'error\':type(error).__name__+\': \'+str(error),\'finished_utc\':utc(),\n            \'resume\':\'jalankan notebook32 lagi; cache memeriksa isi, kode dan konfigurasi; jangan hapus KONTROL/BANK/MODELS/PREDIKSI\'})\n        raise\n'

_transport32=types.ModuleType('_transport32')
exec(compile(RUNTIME32_SOURCE,'_transport32','exec'),_transport32.__dict__)


## Ketidakpastian dan cache

Ukuran utama Pendikom adalah rerata balanced accuracy empat dimensi, dengan
accuracy, macro-F1, confusion matrices, reference counts dan coverage. Prediksi
unresolved dihitung sebagai abstention/kesalahan; referensi teacher LMS yang
unresolved tidak diperlakukan sebagai kutub ILS dan coverage-nya dilaporkan.
Teacher direct bernilai 100% fidelity ketika resolved secara definisi.

Bootstrap 10000 mengambil ulang KURSUS, mempertahankan seluruh peserta, empat
dimensi, metode dan run dalam kelompok yang sama. Untuk stokastik, hitung metrik
setiap run lalu rerata, tanpa voting atau best-seed. Bobot baru dibuat untuk
peserta protokol baru, bukan memakai bobot 139 peserta dari tahap sebelumnya.
CI 95 bersyarat pada fitted folds dan runs yang tersimpan; variasi training baru
tidak tercakup penuh. Fold dengan train overlap bukan dataset independen.

Tiga keluarga kontras meanBA: tujuh initializer, tiga operator profil dan tiga
classifier. Masing-masing menyediakan paired 95 CI dan interval keluarga dengan
Bonferroni. CI metrik lain deskriptif. Missing pole menyebabkan BA unavailable;
jumlah draw bootstrap valid disimpan. Per-course BA/F1 hanya dilaporkan jika
kedua kutub referensi tersedia; kursus kecil tetap masuk denominator keseluruhan.

Cache memeriksa isi artefak, kode, versi, train aktual, seed dan konfigurasi. Fit
berdasarkan train dipisah dari prediksi test. Jalankan ulang setelah terputus;
jangan hapus KONTROL/BANK/MODELS/PREDIKSI. force=True menghitung ulang dan
mengarsipkan catatan/artefak sebelumnya di dalam tahap 32. Autosave output Colab
ditangani dengan pembacaan JSON stabil; perubahan kode tetap ditolak.


In [5]:
HASIL_32=_transport32.run(ROOT_32,KONFIGURASI_32,RUNTIME32_SOURCE,CAPSULES_32)
display(HASIL_32['summary'].loc[HASIL_32['summary'].dimension.eq('OVERALL')])
display(HASIL_32['paired'].loc[HASIL_32['paired'].metric.eq('mean_balanced_accuracy')])
print(HASIL_32['status'])
print('Folder hasil:',HASIL_32['output'])


Konteks 1 / 11 PENDIKOM/COURSE_GROUP/F01
   CARE-KM selesai; {'FIT': 1, 'CACHE': 0}
   Cao selesai; {'FIT': 2, 'CACHE': 0}
   Bai selesai; {'FIT': 3, 'CACHE': 0}
   BK-Modes selesai; {'FIT': 4, 'CACHE': 0}
   K-Modes++ selesai; {'FIT': 19, 'CACHE': 0}
   Random selesai; {'FIT': 34, 'CACHE': 0}
   Huang selesai; {'FIT': 49, 'CACHE': 0}
   HR-2023 selesai; {'FIT': 64, 'CACHE': 0}
Konteks 2 / 11 PENDIKOM/COURSE_GROUP/F02
   CARE-KM selesai; {'FIT': 65, 'CACHE': 0}
   Cao selesai; {'FIT': 66, 'CACHE': 0}
   Bai selesai; {'FIT': 67, 'CACHE': 0}
   BK-Modes selesai; {'FIT': 68, 'CACHE': 0}
   K-Modes++ selesai; {'FIT': 83, 'CACHE': 0}
   Random selesai; {'FIT': 98, 'CACHE': 0}
   Huang selesai; {'FIT': 113, 'CACHE': 0}
   HR-2023 selesai; {'FIT': 128, 'CACHE': 0}
Konteks 3 / 11 PENDIKOM/COURSE_GROUP/F03
   CARE-KM selesai; {'FIT': 129, 'CACHE': 0}
   Cao selesai; {'FIT': 130, 'CACHE': 0}
   Bai selesai; {'FIT': 131, 'CACHE': 0}
   BK-Modes selesai; {'FIT': 132, 'CACHE': 0}
   K-Modes++ seles

,cohort,protocol,pathway,dimension,n_test,n_courses,n_runs,reference,coverage,full_profile_coverage,...,balanced_accuracy,balanced_accuracy_ci_low,balanced_accuracy_ci_high,balanced_accuracy_valid_bootstrap,balanced_accuracy_seed_sd,macro_f1,macro_f1_ci_low,macro_f1_ci_high,macro_f1_valid_bootstrap,macro_f1_seed_sd
4,PENDIKOM,COURSE_GROUP,CARE-KM_DIRECT,OVERALL,461,50,1,INDEPENDENT_ILS,0.996204,0.984816,...,0.787239,0.751976,0.821350,10000,0.000000,0.801753,0.763253,0.834379,10000,0.000000
9,PENDIKOM,COURSE_GROUP,CARE_MEMBER_ONLY,OVERALL,461,50,1,INDEPENDENT_ILS,0.991323,0.965293,...,0.785221,0.749411,0.818719,10000,0.000000,0.804540,0.767258,0.836040,10000,0.000000
14,PENDIKOM,COURSE_GROUP,CARE_MEMBER_FIRST,OVERALL,461,50,1,INDEPENDENT_ILS,0.996204,0.984816,...,0.788520,0.754764,0.820841,10000,0.000000,0.805418,0.768828,0.836561,10000,0.000000
19,PENDIKOM,COURSE_GROUP,Cao_DIRECT,OVERALL,461,50,1,INDEPENDENT_ILS,0.998373,0.993492,...,0.764146,0.734466,0.791239,10000,0.000000,0.779357,0.746389,0.806805,10000,0.000000
24,PENDIKOM,COURSE_GROUP,Bai_DIRECT,OVERALL,461,50,1,INDEPENDENT_ILS,1.000000,1.000000,...,0.796145,0.757425,0.830563,10000,0.000000,0.801612,0.761239,0.832815,10000,0.000000
29,PENDIKOM,COURSE_GROUP,BK-Modes_DIRECT,OVERALL,461,50,1,INDEPENDENT_ILS,0.989696,0.958785,...,0.739745,0.704140,0.770507,10000,0.000000,0.734115,0.693780,0.765799,10000,0.000000
34,PENDIKOM,COURSE_GROUP,K-Modes++_DIRECT,OVERALL,461,50,15,INDEPENDENT_ILS,0.996529,0.986262,...,0.745750,0.719768,0.769147,10000,0.010867,0.742019,0.711807,0.765612,10000,0.013366
39,PENDIKOM,COURSE_GROUP,Random_DIRECT,OVERALL,461,50,15,INDEPENDENT_ILS,0.996457,0.985828,...,0.729305,0.705787,0.750316,10000,0.018803,0.722271,0.692062,0.745336,10000,0.022778
44,PENDIKOM,COURSE_GROUP,Huang_DIRECT,OVERALL,461,50,15,INDEPENDENT_ILS,0.995915,0.983659,...,0.705692,0.685052,0.724686,10000,0.018479,0.702206,0.675523,0.723260,10000,0.021540
49,PENDIKOM,COURSE_GROUP,HR-2023_DIRECT,OVERALL,461,50,15,INDEPENDENT_ILS,0.996638,0.986551,...,0.707129,0.688897,0.724002,10000,0.018127,0.704032,0.679798,0.723099,10000,0.021159


,cohort,protocol,family,reference,comparison,metric,point_difference,ci_low,ci_high,valid_bootstrap,n_family_contrasts,family95_ci_low,family95_ci_high,multiplicity
1,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_Random_DIRECT,mean_balanced_accuracy,0.057934,0.031791,0.082446,10000,7,0.023069,0.090593,Bonferroni percentile family interval for prim...
4,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_Huang_DIRECT,mean_balanced_accuracy,0.081548,0.055358,0.106995,10000,7,0.047497,0.115578,Bonferroni percentile family interval for prim...
7,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_Cao_DIRECT,mean_balanced_accuracy,0.023093,-0.003460,0.047544,10000,7,-0.014262,0.057420,Bonferroni percentile family interval for prim...
10,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_Bai_DIRECT,mean_balanced_accuracy,-0.008906,-0.039936,0.018349,10000,7,-0.052717,0.026966,Bonferroni percentile family interval for prim...
13,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_K-Modes++_DIRECT,mean_balanced_accuracy,0.041489,0.011983,0.067300,10000,7,0.002268,0.075916,Bonferroni percentile family interval for prim...
16,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_HR-2023_DIRECT,mean_balanced_accuracy,0.080110,0.054057,0.106100,10000,7,0.046419,0.115320,Bonferroni percentile family interval for prim...
19,PENDIKOM,COURSE_GROUP,initializer,INDEPENDENT_ILS,CARE_minus_BK-Modes_DIRECT,mean_balanced_accuracy,0.047495,0.013502,0.080714,10000,7,0.000264,0.093258,Bonferroni percentile family interval for prim...
22,PENDIKOM,COURSE_GROUP,profile_operator,INDEPENDENT_ILS,CARE_minus_BEHAVIORAL_DIRECT,mean_balanced_accuracy,0.080525,0.042652,0.109639,10000,3,0.032827,0.116026,Bonferroni percentile family interval for prim...
25,PENDIKOM,COURSE_GROUP,profile_operator,INDEPENDENT_ILS,CARE_minus_CARE_MEMBER_ONLY,mean_balanced_accuracy,0.002018,-0.005196,0.007863,10000,3,-0.006862,0.009211,Bonferroni percentile family interval for prim...
28,PENDIKOM,COURSE_GROUP,profile_operator,INDEPENDENT_ILS,CARE_minus_CARE_MEMBER_FIRST,mean_balanced_accuracy,-0.001280,-0.008087,0.004088,10000,3,-0.009803,0.005181,Bonferroni percentile family interval for prim...


{'status': 'selesai', 'full_protocol': True, 'course_protocol_full': True, 'temporal_protocol_full': True, 'completed_fit_jobs': 704, 'planned_fit_jobs': 704, 'fit_counts': {'FIT': 704, 'CACHE': 0}, 'failed_fit_jobs': 0, 'model_counts': {'MODEL_FIT': 33, 'CACHE': 0}, 'model_estimators': 78, 'failed_model_estimators': 0, 'n_contexts': 11, 'n_prediction_rows': 506380, 'n_methods': 8, 'k': 16, 'q': 16, 'independent_ILS_training_targets': 0, 'failed_prediction_routes': 0, 'temporal_audit': [{'cohort': 'PENDIKOM', 'protocol': 'TEMPORAL', 'status': 'ELIGIBLE', 'cutoff_utc': '2021-07-01T00:00:00+00:00', 'train_n': 346, 'test_n': 115, 'excluded_n': 0, 'invalid_timestamp_n': 0, 'purged_training_course_overlap_n': 0, 'reason': 'Fixed calendar cutoff; whole observation windows; no score-based cutoff search; no early-prediction claim.'}, {'cohort': 'LMS', 'protocol': 'TEMPORAL', 'status': 'AUDIT_ONLY', 'reason': 'Existing numeric reconstruction uses full-course P95 observation windows; no forward-

## Keluaran dan penggunaan di manuskrip

- `AUDIT/`: seleksi 565→461, distribusi kursus, rekonstruksi sumber, kelayakan waktu,
  seluruh fit attempts dan jumlah target train classifier.
- `SPLIT/`: keanggotaan train/test, kategori baru, threshold, Q75, mask, kamus evidence,
  serta bukti nol overlap tiap konteks; split lama tidak diubah.
- `BANK/`: mode awal/final, assignment train, mapping, evidence, dan target train per run.
- `MODELS/`, `PREDIKSI/`: model, assignment test, seluruh prediksi peserta/course/fold,
  termasuk failure dan kasus unresolved.
- `EVALUASI/`: tabel per dimensi/run, kursus, reference counts, confusion, paired contrasts.
- `STATISTIK/`: bobot bootstrap kursus dan bank metrik untuk analisis gabungan.
- `TABEL/04_Pendikom_ILS.csv`: hasil utama antarkursus dan temporal vs ILS.
- `TABEL/05_LMS_classifier_fidelity.csv`: hasil LMS untuk jalur utama/classifier.
- `TABEL/06_LMS_semantic_concordance_diagnostic.csv`: concordance baseline terhadap
  teacher CARE; bukan ranking validitas ILS.
- `GAMBAR/`: PNG/PDF dengan font 11 pt; `PROTOKOL/`,`KONTROL/`: manifest, checkpoint, status.
